[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/calculus/12_hessian_jacobian_curvature/exercises.ipynb)

# Module 12 — Hessian, Jacobian, and Curvature: Exercises

---

Every boxed answer below that is numeric or algorithmic is recomputed by a code cell that runs and asserts agreement. Run this preamble first.

In [1]:
import numpy as np
import sympy as sp

rng = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)


def softmax(z):
    e = np.exp(z - np.max(z))
    return e / e.sum()


def report(name, value, target, tol=1e-10):
    """Print a residual and assert it sits at rounding-noise level."""
    r = float(np.max(np.abs(np.asarray(value, dtype=float) - np.asarray(target, dtype=float))))
    print(f"{name:52s} residual = {r:.3e}")
    assert r < tol, (name, r)

## L0 — Concept Checks

### Problem L0.1 — Linear Map Jacobian vs Scalar Derivative
**Source:** Adapted from Stewart, *Multivariable Calculus* (Ch. 14) & MIT 18.02.  
**Problem Statement:**  
Consider a linear mapping $f: \mathbb{R}^n \to \mathbb{R}^m$ defined by $f(x) = A x + b$, where $A \in \mathbb{R}^{m \times n}$ and $b \in \mathbb{R}^m$. Compute the Jacobian matrix $J_f(x)$ at any point $x \in \mathbb{R}^n$, and explain how this generalizes the single-variable derivative of $f(x) = a x + b$.

#### First-Principles Intuition
The Jacobian matrix represents the best local linear approximation of a function near $x$. If the function is already linear, its local linear approximation must be identical to the global linear map itself, independent of the point $x$.

#### Step-by-Step Solution
Write the $i$-th component of $f(x)$:

$$
f_i(x) = \sum_{k=1}^n A_{ik} x_k + b_i
$$

Compute the partial derivative of $f_i$ with respect to $x_j$:

$$
\frac{\partial f_i}{\partial x_j} = \frac{\partial}{\partial x_j} \left( \sum_{k=1}^n A_{ik} x_k + b_i \right) = A_{ij}
$$

By definition of the Jacobian matrix $(J_f(x))_{ij} = \frac{\partial f_i}{\partial x_j}$, every entry of $J_f(x)$ equals $A_{ij}$. Thus:

$$
\mathbf{d}f = J_f(x) \mathbf{d}x = A \mathbf{d}x
$$

In single-variable calculus, for $f(x) = ax + b$, $f'(x) = a$. Here, for $f(x) = Ax + b$, $J_f(x) = A$.

$$
\boxed{J_f(x) = A}
$$

#### Key Insight / Takeaway
The Jacobian of an affine mapping $Ax + b$ is the constant matrix $A$. Nonlinear functions have Jacobians that vary with position $x$, acting as local linearized approximations.

**Verification.** The cell below recomputes the boxed answer of Problem L0.1 and asserts agreement.

In [2]:
# J_f(x) = A for f(x) = Ax + b, at any x, checked against a finite-difference Jacobian.
A = rng.normal(size=(3, 4))
b = rng.normal(size=3)
f = lambda v: A @ v + b

def jac_fd(fun, v, h=1e-6):
    n = v.size
    m = fun(v).size
    J = np.zeros((m, n))
    for j in range(n):
        e = np.zeros(n); e[j] = h
        J[:, j] = (fun(v + e) - fun(v - e)) / (2 * h)
    return J

for x0 in (np.zeros(4), rng.normal(size=4), 10 * rng.normal(size=4)):
    report("L0.1  finite-difference J_f(x) vs A", jac_fd(f, x0), A, tol=1e-8)
print("\nJ_f is the same matrix A at every point, as the boxed answer states.")

L0.1  finite-difference J_f(x) vs A                  residual = 2.015e-10
L0.1  finite-difference J_f(x) vs A                  residual = 2.015e-10
L0.1  finite-difference J_f(x) vs A                  residual = 1.186e-09

J_f is the same matrix A at every point, as the boxed answer states.


---

### Problem L0.2 — Geometric Interpretation of $\det(J)$ in 2D Polar Coordinates
**Source:** Adapted from Apostol, *Calculus Vol II* & Spivak, *Calculus on Manifolds*.  
**Problem Statement:**  
The transformation from polar coordinates $(r, \theta)$ to Cartesian coordinates $(x, y)$ is given by $x = r \cos \theta$ and $y = r \sin \theta$. Compute the Jacobian matrix $J$ and its determinant $\det(J)$. Explain why $\det(J) = r$ geometrically.

#### First-Principles Intuition
An infinitesimal rectangle in polar space $[r, r + dr] \times [\theta, \theta + d\theta]$ transforms into a curved sector in Cartesian space. The radial edge has length $dr$, while the arc length along the angular direction is $r d\theta$. The infinitesimal area is thus $r \, dr \, d\theta$. The Jacobian determinant measures this local area scaling factor.

#### Step-by-Step Solution
Define vector function $f(r, \theta) = (r \cos \theta, r \sin \theta)^T$.
Compute partial derivatives:

$$
J = \begin{bmatrix} \frac{\partial x}{\partial r} & \frac{\partial x}{\partial \theta} \\[4pt] \frac{\partial y}{\partial r} & \frac{\partial y}{\partial \theta} \end{bmatrix} = \begin{bmatrix} \cos \theta & -r \sin \theta \\ \sin \theta & r \cos \theta \end{bmatrix}
$$

Compute the determinant:

$$
\det(J) = (\cos \theta)(r \cos \theta) - (-r \sin \theta)(\sin \theta) = r \cos^2 \theta + r \sin^2 \theta = r (\cos^2 \theta + \sin^2 \theta) = r
$$

$$
\boxed{\det(J) = r, \quad \text{so } dA = \lvert\det(J)\rvert \, dr \, d\theta = r \, dr \, d\theta}
$$

#### Key Insight / Takeaway
The factor $r$ in $dA = r \, dr \, d\theta$ arises purely from the Jacobian determinant, quantifying how distance from the origin scales differential arc lengths.

**Verification.** The cell below recomputes the boxed answer of Problem L0.2 and asserts agreement.

In [3]:
# Polar map: det J = r, and the image-cell area matches r*dr*dtheta.
r_s, th_s = sp.symbols("r theta", positive=True)
J_pol = sp.Matrix([r_s * sp.cos(th_s), r_s * sp.sin(th_s)]).jacobian([r_s, th_s])
det_pol = sp.simplify(J_pol.det())
print("symbolic det J =", det_pol)
assert sp.simplify(det_pol - r_s) == 0

def shoelace(p):
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

r_a, dr, t_a, dt = 1.3, 1e-3, 0.7, 1e-3
ts = np.linspace(t_a, t_a + dt, 60)
poly = np.concatenate([np.stack([r_a * np.cos(ts), r_a * np.sin(ts)], 1),
                       np.stack([(r_a + dr) * np.cos(ts[::-1]), (r_a + dr) * np.sin(ts[::-1])], 1)])
area = shoelace(poly)
pred = (r_a + dr / 2) * dr * dt
print(f"measured cell area = {area:.12e}   r*dr*dtheta = {pred:.12e}")
assert abs(area - pred) / pred < 1e-6

symbolic det J = r
measured cell area = 1.300500002799e-06   r*dr*dtheta = 1.300500000000e-06


---

### Problem L0.3 — Eigenspace Interpretation of Hessian Matrix
**Source:** Adapted from Strang, *Linear Algebra and Learning from Data*.  
**Problem Statement:**  
Let $f: \mathbb{R}^2 \to \mathbb{R}$ have a critical point at $(0,0)$ with Hessian:

$$
H = \begin{bmatrix} 3 & 0 \\ 0 & -2 \end{bmatrix}
$$

Identify the principal curvature directions, the corresponding eigenvalues, and classify the critical point.

#### First-Principles Intuition
For a diagonal Hessian, the coordinate axes align directly with the eigenvectors. The eigenvalues give the exact second-derivative (curvature) along those principal directions.

#### Step-by-Step Solution
The Hessian is diagonal:

$$
H = \begin{bmatrix} 3 & 0 \\ 0 & -2 \end{bmatrix}
$$
- Eigenvalue $\lambda_1 = 3 \gt 0$ along eigenvector $v_1 = (1, 0)^T$ (x-axis).
- Eigenvalue $\lambda_2 = -2 \lt 0$ along eigenvector $v_2 = (0, 1)^T$ (y-axis).

Along the x-axis ($y=0$), $f(x,0) \approx f(0,0) + \frac{3}{2} x^2$, bending upward like a valley.  
Along the y-axis ($x=0$), $f(0,y) \approx f(0,0) - y^2$, bending downward like a ridge.  
Since the eigenvalues have opposite signs ($\lambda_1 \gt 0, \lambda_2 \lt 0$), the matrix is indefinite.

$$
\boxed{\text{Principal directions: } e_1=(1,0)^T \text{ (}\lambda=3\text{, upward), } e_2=(0,1)^T \text{ (}\lambda=-2\text{, downward). Critical point: Saddle Point.}}
$$

#### Key Insight / Takeaway
Hessian eigenvectors define orthogonal directions of principal curvature, while eigenvalues specify the magnitude and sign of concavity along those directions.

**Verification.** The cell below recomputes the boxed answer of Problem L0.3 and asserts agreement.

In [4]:
# Diagonal Hessian: eigenpairs are the axes, and the signs give the verdict.
H = np.array([[3.0, 0.0], [0.0, -2.0]])
lam, U = np.linalg.eigh(H)
print("eigenvalues :", lam)
print("eigenvectors (columns):\n", U)
report("L0.3  H e1 - 3 e1", H @ np.array([1.0, 0.0]), np.array([3.0, 0.0]))
report("L0.3  H e2 + 2 e2", H @ np.array([0.0, 1.0]), np.array([0.0, -2.0]))
assert lam.min() < 0 < lam.max()
print("\nmixed signs -> indefinite -> saddle point, as boxed.")

eigenvalues : [-2.  3.]
eigenvectors (columns):
 [[0. 1.]
 [1. 0.]]
L0.3  H e1 - 3 e1                                    residual = 0.000e+00
L0.3  H e2 + 2 e2                                    residual = 0.000e+00

mixed signs -> indefinite -> saddle point, as boxed.


---

### Problem L0.4 — Saddle Points and Rayleigh Quotient Sign Flip
**Source:** Adapted from Boyd & Vandenberghe, *Convex Optimization* (Ch. 3).  
**Problem Statement:**  
Let $H \in \mathbb{R}^{n \times n}$ be an indefinite symmetric matrix with eigenvalues $\lambda_{\min} \lt 0 \lt \lambda_{\max}$. Show that there exist direction vectors $u, v \in \mathbb{R}^n$ such that the Rayleigh quotient $R_H(u) \gt 0$ and $R_H(v) \lt 0$.

#### First-Principles Intuition
An indefinite Hessian has at least one positive and one negative eigenvalue. Evaluating the quadratic form along the respective eigenvectors isolates those single-eigenvalue curvatures.

#### Step-by-Step Solution
Let $v_{\max}$ be the normalized eigenvector corresponding to $\lambda_{\max} \gt 0$ ($H v_{\max} = \lambda_{\max} v_{\max}$, $v_{\max}^T v_{\max} = 1$).  
Let $v_{\min}$ be the normalized eigenvector corresponding to $\lambda_{\min} \lt 0$ ($H v_{\min} = \lambda_{\min} v_{\min}$, $v_{\min}^T v_{\min} = 1$).

Compute the Rayleigh quotient:

$$
R_H(v_{\max}) = \frac{v_{\max}^T H v_{\max}}{v_{\max}^T v_{\max}} = \frac{v_{\max}^T (\lambda_{\max} v_{\max})}{1} = \lambda_{\max} \gt 0
$$

$$
R_H(v_{\min}) = \frac{v_{\min}^T H v_{\min}}{v_{\min}^T v_{\min}} = \frac{v_{\min}^T (\lambda_{\min} v_{\min})}{1} = \lambda_{\min} \lt 0
$$

Thus, setting $u = v_{\max}$ gives positive curvature ($R_H(u) \gt 0$), and setting $v = v_{\min}$ gives negative curvature ($R_H(v) \lt 0$).

$$
\boxed{R_H(v_{\max}) = \lambda_{\max} \gt 0 \quad \text{and} \quad R_H(v_{\min}) = \lambda_{\min} \lt 0}
$$

#### Key Insight / Takeaway
A saddle point is characterized by directions of positive curvature (along which the function increases) and negative curvature (along which it decreases).

**Verification.** The cell below recomputes the boxed answer of Problem L0.4 and asserts agreement.

In [5]:
# An indefinite symmetric H attains both signs of the Rayleigh quotient at its
# extreme eigenvectors, and no direction leaves [lambda_min, lambda_max].
Q = np.linalg.qr(rng.normal(size=(5, 5)))[0]
lam_true = np.array([-3.0, -0.5, 0.25, 1.0, 4.0])
H = Q @ np.diag(lam_true) @ Q.T
H = 0.5 * (H + H.T)
lam, U = np.linalg.eigh(H)
R = lambda v: float(v @ H @ v / (v @ v))
print(f"R(v_min) = {R(U[:, 0]):+.6f}   lambda_min = {lam.min():+.6f}")
print(f"R(v_max) = {R(U[:, -1]):+.6f}   lambda_max = {lam.max():+.6f}")
report("L0.4  R(v_min) - lambda_min", R(U[:, 0]), lam.min())
report("L0.4  R(v_max) - lambda_max", R(U[:, -1]), lam.max())
vals = [R(v) for v in rng.normal(size=(20000, 5))]
assert lam.min() - 1e-12 <= min(vals) and max(vals) <= lam.max() + 1e-12
print(f"\n20000 random directions all landed in [{lam.min():.4f}, {lam.max():.4f}].")

R(v_min) = -3.000000   lambda_min = -3.000000
R(v_max) = +4.000000   lambda_max = +4.000000
L0.4  R(v_min) - lambda_min                          residual = 4.441e-16
L0.4  R(v_max) - lambda_max                          residual = 8.882e-16

20000 random directions all landed in [-3.0000, 4.0000].


---

### Problem L0.5 — Why Gradient Descent Oscillates on High-Condition Number Hessians
**Source:** Adapted from Polyak, *Introduction to Optimization* (1987) & Nocedal & Wright.  
**Problem Statement:**  
Consider $f(x_1, x_2) = \frac{1}{2}(100 x_1^2 + x_2^2)$. Compute the Hessian $H$, its eigenvalues, condition number $\kappa(H)$, and explain why gradient descent with step size $\alpha = 0.019$ oscillates wildly in $x_1$ while progressing slowly in $x_2$.

#### First-Principles Intuition
The function forms a narrow elliptical ravine. The curvature along $x_1$ is 100 times steeper than along $x_2$. A step size suitable for $x_1$ must be small ($\alpha \lt \frac{2}{100} = 0.02$), which makes progress along $x_2$ painstakingly slow.

#### Step-by-Step Solution
Compute gradient and Hessian:

$$
\nabla f(x_1, x_2) = \begin{bmatrix} 100 x_1 \\ x_2 \end{bmatrix}, \quad H = \begin{bmatrix} 100 & 0 \\ 0 & 1 \end{bmatrix}
$$

Eigenvalues are $\lambda_1 = 100$ and $\lambda_2 = 1$.  
The condition number is:

$$
\kappa(H) = \frac{\lambda_{\max}}{\lambda_{\min}} = \frac{100}{1} = 100
$$

The gradient descent update step is:

$$
x_1^{(k+1)} = x_1^{(k)} - \alpha (100 x_1^{(k)}) = (1 - 100\alpha) x_1^{(k)}
$$

$$
x_2^{(k+1)} = x_2^{(k)} - \alpha (1 x_2^{(k)}) = (1 - \alpha) x_2^{(k)}
$$

For $\alpha = 0.019$:
- $1 - 100(0.019) = 1 - 1.9 = -0.9$. Since $\lvert -0.9 \rvert \lt 1$, $x_1$ flips sign every step (oscillates).
- $1 - \alpha = 1 - 0.019 = 0.981$. $x_2$ shrinks by only $1.9\%$ per step (extremely slow progress).

$$
\boxed{H = \begin{bmatrix} 100 & 0 \\ 0 & 1 \end{bmatrix}, \quad \kappa(H) = 100. \text{ $x_1$ contraction factor } -0.9 \text{ (oscillation), } x_2 \text{ factor } 0.981 \text{ (sluggish).}}
$$

#### Key Insight / Takeaway
Large condition numbers $\kappa(H) \gg 1$ restrict maximum stable step sizes to $\alpha \lt \frac{2}{\lambda_{\max}}$, choking convergence along low-curvature directions ($\lambda_{\min}$).

**Verification.** The cell below recomputes the boxed answer of Problem L0.5 and asserts agreement.

In [6]:
# f = 0.5*(100 x1^2 + x2^2): kappa = 100, and alpha = 0.019 gives the two factors.
H = np.array([[100.0, 0.0], [0.0, 1.0]])
lam = np.linalg.eigvalsh(H)
kappa = lam.max() / lam.min()
alpha = 0.019
factors = 1 - alpha * lam           # ascending: [1 - alpha*1, 1 - alpha*100]
print(f"eigenvalues = {lam}   kappa = {kappa:g}")
print(f"contraction factor along x2 (lambda=1)   = {factors[0]:+.4f}")
print(f"contraction factor along x1 (lambda=100) = {factors[1]:+.4f}")
report("L0.5  kappa", kappa, 100.0)
report("L0.5  x1 factor", factors[1], -0.9)
report("L0.5  x2 factor", factors[0], 0.981)

x = np.array([1.0, 1.0])
signs = []
for _ in range(6):
    x = x - alpha * (H @ x)
    signs.append(np.sign(x[0]))
print("\nsign of x1 over six steps:", signs, "-> alternates, i.e. oscillation")
assert all(signs[k] != signs[k + 1] for k in range(len(signs) - 1))
print(f"after 6 steps x2 has only shrunk to {0.981 ** 6:.4f} of its start")

eigenvalues = [  1. 100.]   kappa = 100
contraction factor along x2 (lambda=1)   = +0.9810
contraction factor along x1 (lambda=100) = -0.9000
L0.5  kappa                                          residual = 0.000e+00
L0.5  x1 factor                                      residual = 1.110e-16
L0.5  x2 factor                                      residual = 0.000e+00

sign of x1 over six steps: [np.float64(-1.0), np.float64(1.0), np.float64(-1.0), np.float64(1.0), np.float64(-1.0), np.float64(1.0)] -> alternates, i.e. oscillation
after 6 steps x2 has only shrunk to 0.8913 of its start


---

### Problem L0.6 — Multivariable Newton's Step as Quadratic Minimizer
**Source:** Adapted from Nocedal & Wright, *Numerical Optimization* (Ch. 3).  
**Problem Statement:**  
Let $f: \mathbb{R}^n \to \mathbb{R}$ be $C^2$ with $\nabla f(x_k) = g_k$ and $H_f(x_k) = H_k \succ 0$. Show that the Newton step $p_k = -H_k^{-1} g_k$ uniquely minimizes the local quadratic model $m_k(p) = f(x_k) + g_k^T p + \frac{1}{2} p^T H_k p$.

#### First-Principles Intuition
Newton's method replaces the complicated objective function with its second-order Taylor expansion and jumps directly to the minimum of that quadratic approximation.

#### Step-by-Step Solution
Differentiate $m_k(p)$ with respect to the step vector $p$:

$$
\nabla_p m_k(p) = \nabla_p \left( f(x_k) + g_k^T p + \frac{1}{2} p^T H_k p \right) = g_k + H_k p
$$

Setting $\nabla_p m_k(p) = \mathbf{0}$:

$$
g_k + H_k p = \mathbf{0} \implies H_k p = -g_k \implies p = -H_k^{-1} g_k
$$

Compute the second derivative (Hessian) of $m_k(p)$ with respect to $p$:

$$
\nabla_p^2 m_k(p) = H_k
$$

Since $H_k \succ 0$, the quadratic model $m_k(p)$ is strictly convex and $p_k = -H_k^{-1} g_k$ is the unique global minimizer of $m_k(p)$.

$$
\boxed{p_k = -H_k^{-1} g_k \quad \text{uniquely minimizes } m_k(p)}
$$

#### Key Insight / Takeaway
Unlike gradient descent which only uses local slope, Newton's step utilizes second-order curvature $H_k$ to scale and rotate the update vector directly toward the quadratic model's minimizer.

**Verification.** The cell below recomputes the boxed answer of Problem L0.6 and asserts agreement.

In [7]:
# p = -H^{-1} g is the unique stationary point of the quadratic model, and no other
# step attains a smaller model value.
n = 5
B = rng.normal(size=(n, n))
H = B @ B.T + n * np.eye(n)          # symmetric positive definite
g = rng.normal(size=n)
p_star = -np.linalg.solve(H, g)
m = lambda p: float(g @ p + 0.5 * p @ H @ p)
report("L0.6  gradient of the model at p*", g + H @ p_star, np.zeros(n))
best = min(m(p_star + 1e-2 * d) for d in rng.normal(size=(20000, n)))
print(f"model at p*            = {m(p_star):.10f}")
print(f"best of 20000 nearby p = {best:.10f}")
assert m(p_star) < best
print("\nH is positive definite, so p* is the unique global minimiser of the model.")

L0.6  gradient of the model at p*                    residual = 6.939e-18


model at p*            = -0.0156570994
best of 20000 nearby p = -0.0156437745

H is positive definite, so p* is the unique global minimiser of the model.


---

### Problem L0.7 — Softmax Jacobian Rank Deficiency along All-Ones Vector
**Source:** Adapted from Goodfellow et al., *Deep Learning* (Ch. 4).  
**Problem Statement:**  
Let $S(z) \in \mathbb{R}^n$ be the Softmax function with Jacobian $J = \operatorname{diag}(S) - S S^T$. Show that $J \mathbf{1} = \mathbf{0}$, where $\mathbf{1} = (1, 1, \dots, 1)^T \in \mathbb{R}^n$. Explain the geometric meaning of this result.

#### First-Principles Intuition
Adding a constant scalar $c$ to all logits ($z + c\mathbf{1}$) does not change the probabilities ($S(z + c\mathbf{1}) = S(z)$). Thus, the derivative of Softmax along the direction $\mathbf{1}$ must be identically zero.

#### Step-by-Step Solution
Evaluate $J \mathbf{1}$:

$$
J \mathbf{1} = (\operatorname{diag}(S) - S S^T) \mathbf{1} = \operatorname{diag}(S) \mathbf{1} - S (S^T \mathbf{1})
$$

Note that $\operatorname{diag}(S) \mathbf{1} = (S_1, S_2, \dots, S_n)^T = S$.  
Since Softmax probabilities sum to 1, $S^T \mathbf{1} = \sum_{i=1}^n S_i = 1$.

Substitute these into the expression:

$$
J \mathbf{1} = S - S(1) = S - S = \mathbf{0}
$$

Thus, $\mathbf{1}$ lies in the nullspace of $J$, meaning $\operatorname{rank}(J) \le n - 1$.

$$
\boxed{J \mathbf{1} = \mathbf{0}. \text{ Geometric meaning: Softmax is shift-invariant along } \mathbf{1}, \text{ making } J \text{ singular.}}
$$

#### Key Insight / Takeaway
The Softmax function has a 1-dimensional translational symmetry along the vector $\mathbf{1}$. Consequently, its Jacobian is singular with a zero eigenvalue corresponding to eigenvector $\mathbf{1}$.

**Verification.** The cell below recomputes the boxed answer of Problem L0.7 and asserts agreement.

In [8]:
# J 1 = 0 for the softmax Jacobian, and the numerical rank is n - 1.
for n in (3, 6, 10):
    z = rng.normal(size=n) * 2.0
    S = softmax(z)
    J = np.diag(S) - np.outer(S, S)
    report(f"L0.7  J @ 1 for n = {n}", J @ np.ones(n), np.zeros(n), tol=1e-14)
    r = np.linalg.matrix_rank(J, tol=1e-12 * np.abs(J).max())
    print(f"        n = {n:2d}   numerical rank = {r}   (n - 1 = {n - 1})")
    assert r == n - 1
print("\nShift invariance S(z + c1) = S(z) is what puts 1 in the nullspace.")

L0.7  J @ 1 for n = 3                                residual = 2.776e-17
        n =  3   numerical rank = 2   (n - 1 = 2)
L0.7  J @ 1 for n = 6                                residual = 2.776e-17
        n =  6   numerical rank = 5   (n - 1 = 5)
L0.7  J @ 1 for n = 10                               residual = 3.123e-17
        n = 10   numerical rank = 9   (n - 1 = 9)

Shift invariance S(z + c1) = S(z) is what puts 1 in the nullspace.


---

### Problem L0.8 — Positive Semi-Definiteness of Cross-Entropy Softmax Loss Hessian
**Source:** Adapted from Boyd & Vandenberghe, *Convex Optimization* (Ch. 3).  
**Problem Statement:**  
Let $H = \operatorname{diag}(S) - S S^T$ be the Hessian of the multi-class cross-entropy loss with Softmax, where $S_i \gt 0$ and $\sum S_i = 1$. Prove that $v^T H v \ge 0$ for all $v \in \mathbb{R}^n$, confirming that the loss is convex.

#### First-Principles Intuition
The quadratic form $v^T H v$ can be rewritten as the variance of $v$ under the discrete probability distribution defined by $S$. Since variance is non-negative, $v^T H v \ge 0$.

#### Step-by-Step Solution
Expand the quadratic form $v^T H v$:

$$
v^T H v = v^T (\operatorname{diag}(S) - S S^T) v = v^T \operatorname{diag}(S) v - v^T S S^T v = \sum_{i=1}^n S_i v_i^2 - \left( \sum_{i=1}^n S_i v_i \right)^2
$$

Interpret $S$ as a discrete probability distribution over $\{1, 2, \dots, n\}$ with $\mathbb{P}(X = i) = S_i$.  
Then $\sum_{i=1}^n S_i v_i^2 = \mathbb{E}_S[v^2]$ and $\sum_{i=1}^n S_i v_i = \mathbb{E}_S[v]$.

Thus:

$$
v^T H v = \mathbb{E}_S[v^2] - (\mathbb{E}_S[v])^2 = \operatorname{Var}_S(v)
$$

By probability theory, variance is always non-negative: $\operatorname{Var}_S(v) \ge 0$.  
Therefore, $v^T H v \ge 0$ for all $v \in \mathbb{R}^n$, which proves $H \succeq 0$.

$$
\boxed{v^T H v = \operatorname{Var}_S(v) \ge 0 \implies H \succeq 0 \text{ (Cross-Entropy loss is convex in logits } z\text{)}}
$$

#### Key Insight / Takeaway
The Hessian of Softmax cross-entropy loss represents a probability variance operator. Its positive semi-definiteness guarantees global convexity without local non-global minima.

**Verification.** The cell below recomputes the boxed answer of Problem L0.8 and asserts agreement.

In [9]:
# v^T H v equals Var_S(v) >= 0, checked on random S and v, including the equality case.
for _ in range(5):
    n = int(rng.integers(2, 9))
    S = softmax(rng.normal(size=n) * 1.5)
    v = rng.normal(size=n)
    H = np.diag(S) - np.outer(S, S)
    quad = float(v @ H @ v)
    var = float(S @ v**2 - (S @ v) ** 2)
    report(f"L0.8  v^T H v - Var_S(v)  (n = {n})", quad, var, tol=1e-12)
    assert quad >= -1e-15
S = softmax(rng.normal(size=7))
H = np.diag(S) - np.outer(S, S)
print(f"\nequality case v = 1: v^T H v = {float(np.ones(7) @ H @ np.ones(7)):.3e} (exactly the flat direction)")
print(f"smallest eigenvalue of H = {np.linalg.eigvalsh(H).min():.3e}  -> H is PSD, not PD")

L0.8  v^T H v - Var_S(v)  (n = 2)                    residual = 0.000e+00
L0.8  v^T H v - Var_S(v)  (n = 6)                    residual = 1.110e-16
L0.8  v^T H v - Var_S(v)  (n = 4)                    residual = 5.551e-17
L0.8  v^T H v - Var_S(v)  (n = 7)                    residual = 2.220e-16
L0.8  v^T H v - Var_S(v)  (n = 7)                    residual = 6.661e-16

equality case v = 1: v^T H v = -8.934e-17 (exactly the flat direction)
smallest eigenvalue of H = -6.510e-18  -> H is PSD, not PD


---

## L1 — Foundations

### Problem L1.1 — Calculating Jacobian Matrix & Determinant of 3D Spherical Coordinates
**Source:** Adapted from Stewart, *Multivariable Calculus* (Ch. 15) & Demidovich No. 3156.  
**Problem Statement:**  
The transformation from spherical coordinates $(\rho, \theta, \phi)$ to Cartesian coordinates $(x, y, z)$ is:

$$
x = \rho \sin \phi \cos \theta, \quad y = \rho \sin \phi \sin \theta, \quad z = \rho \cos \phi
$$

Compute the $3 \times 3$ Jacobian matrix $J$ and show that $\det(J) = \rho^2 \sin \phi$.

#### First-Principles Intuition
The spherical volume element $dV = dx \, dy \, dz$ transforms into $\rho^2 \sin \phi \, d\rho \, d\phi \, d\theta$. The factor $\rho^2 \sin \phi$ comes directly from the determinant of the linear transformation matrix of local partial derivatives.

#### Step-by-Step Solution
Compute partial derivatives for each row of $J$:

$$
J = \begin{bmatrix}
\frac{\partial x}{\partial \rho} & \frac{\partial x}{\partial \theta} & \frac{\partial x}{\partial \phi} \\[4pt]
\frac{\partial y}{\partial \rho} & \frac{\partial y}{\partial \theta} & \frac{\partial y}{\partial \phi} \\[4pt]
\frac{\partial z}{\partial \rho} & \frac{\partial z}{\partial \theta} & \frac{\partial z}{\partial \phi}
\end{bmatrix} = \begin{bmatrix}
\sin \phi \cos \theta & -\rho \sin \phi \sin \theta & \rho \cos \phi \cos \theta \\
\sin \phi \sin \theta & \rho \sin \phi \cos \theta & \rho \cos \phi \sin \theta \\
\cos \phi & 0 & -\rho \sin \phi
\end{bmatrix}
$$

Expand $\det(J)$ along the 3rd row:

$$
\det(J) = \cos \phi \cdot \begin{vmatrix} -\rho \sin \phi \sin \theta & \rho \cos \phi \cos \theta \\ \rho \sin \phi \cos \theta & \rho \cos \phi \sin \theta \end{vmatrix} - 0 + (-\rho \sin \phi) \cdot \begin{vmatrix} \sin \phi \cos \theta & -\rho \sin \phi \sin \theta \\ \sin \phi \sin \theta & \rho \sin \phi \cos \theta \end{vmatrix}
$$

Compute the first $2 \times 2$ determinant:

$$
D_1 = (-\rho \sin \phi \sin \theta)(\rho \cos \phi \sin \theta) - (\rho \cos \phi \cos \theta)(\rho \sin \phi \cos \theta) = -\rho^2 \sin \phi \cos \phi (\sin^2 \theta + \cos^2 \theta) = -\rho^2 \sin \phi \cos \phi
$$

Compute the second $2 \times 2$ determinant:

$$
D_2 = (\sin \phi \cos \theta)(\rho \sin \phi \cos \theta) - (-\rho \sin \phi \sin \theta)(\sin \phi \sin \theta) = \rho \sin^2 \phi (\cos^2 \theta + \sin^2 \theta) = \rho \sin^2 \phi
$$

Combine results:

$$
\det(J) = \cos \phi (-\rho^2 \sin \phi \cos \phi) - \rho \sin \phi (\rho \sin^2 \phi) = -\rho^2 \sin \phi \cos^2 \phi - \rho^2 \sin^3 \phi = -\rho^2 \sin \phi (\cos^2 \phi + \sin^2 \phi) = -\rho^2 \sin \phi
$$

Taking absolute value for physical differential volume:

$$
\boxed{\det(J) = -\rho^2 \sin \phi \implies dV = \lvert\det(J)\rvert \, d\rho \, d\phi \, d\theta = \rho^2 \sin \phi \, d\rho \, d\phi \, d\theta}
$$

#### Key Insight / Takeaway
The Jacobian determinant of spherical coordinates is $-\rho^2 \sin \phi$. Its magnitude $\rho^2 \sin \phi$ reflects how spherical volume expands with radial distance $\rho$ and zenith angle $\phi$.

**Verification.** The cell below recomputes the boxed answer of Problem L1.1 and asserts agreement.

In [10]:
# Spherical Jacobian in the column order (rho, theta, phi): det J = -rho^2 sin(phi).
rho, th, ph = sp.symbols("rho theta phi", positive=True)
Xs = sp.Matrix([rho * sp.sin(ph) * sp.cos(th), rho * sp.sin(ph) * sp.sin(th), rho * sp.cos(ph)])
J_sph = Xs.jacobian([rho, th, ph])
det_sph = sp.simplify(J_sph.det())
print("columns ordered (rho, theta, phi):  det J =", det_sph)
assert sp.simplify(det_sph + rho**2 * sp.sin(ph)) == 0

det_swap = sp.simplify(Xs.jacobian([rho, ph, th]).det())
print("columns ordered (rho, phi, theta):  det J =", det_swap)
assert sp.simplify(det_swap - rho**2 * sp.sin(ph)) == 0
print("\nSwapping two columns flips the sign; |det J| = rho^2 sin(phi) either way.")

# Numerical spot check of the whole matrix.
vals = {rho: 1.7, th: 0.6, ph: 1.1}
Jn = np.array(J_sph.subs(vals), dtype=float)
report("L1.1  |det J| vs rho^2 sin(phi)", abs(np.linalg.det(Jn)), 1.7**2 * np.sin(1.1), tol=1e-12)

columns ordered (rho, theta, phi):  det J = -rho**2*sin(phi)
columns ordered (rho, phi, theta):  det J = rho**2*sin(phi)

Swapping two columns flips the sign; |det J| = rho^2 sin(phi) either way.
L1.1  |det J| vs rho^2 sin(phi)                      residual = 4.441e-16


---

### Problem L1.2 — Critical Points & Hessian Classification of $f(x,y) = x^3 - 3xy^2 + y^4$
**Source:** Adapted from Demidovich No. 3201 & Stewart Ch. 14.  
**Problem Statement:**  
Find all critical points of $f(x,y) = x^3 - 3xy^2 + y^4$ and classify them using the Hessian matrix second derivative test.

#### First-Principles Intuition
Critical points occur where $\nabla f = \mathbf{0}$. The second partial derivatives evaluated at those points form the Hessian $H$. Eigenvalues or determinant of $H$ reveal local concavity.

#### Step-by-Step Solution
1. **Find Critical Points ($\nabla f = \mathbf{0}$):**

$$
   f_x = 3x^2 - 3y^2 = 3(x^2 - y^2) = 0 \implies x^2 = y^2 \implies x = \pm y
$$

$$
   f_y = -6xy + 4y^3 = 2y(-3x + 2y^2) = 0
$$

   - Case A: $y = 0 \implies x^2 = 0 \implies x = 0$. Critical point: $(0,0)$.
   - Case B: $x = y \neq 0 \implies 2y(-3y + 2y^2) = 0 \implies -3 + 2y = 0 \implies y = \frac{3}{2}, x = \frac{3}{2}$. Critical point: $(\frac{3}{2}, \frac{3}{2})$.
   - Case C: $x = -y \neq 0 \implies 2y(3y + 2y^2) = 0 \implies 3 + 2y = 0 \implies y = -\frac{3}{2}, x = \frac{3}{2}$. Critical point: $(\frac{3}{2}, -\frac{3}{2})$.

2. **Compute Hessian Matrix:**

$$
   H(x,y) = \begin{bmatrix} f_{xx} & f_{xy} \\ f_{yx} & f_{yy} \end{bmatrix} = \begin{bmatrix} 6x & -6y \\ -6y & -6x + 12y^2 \end{bmatrix}
$$

3. **Classify Point $(0,0)$:**

$$
   H(0,0) = \begin{bmatrix} 0 & 0 \\ 0 & 0 \end{bmatrix} \implies \det(H) = 0 \quad (\text{Inconclusive test})
$$

   Analyze along paths: $f(x,0) = x^3$ (changes sign near $x=0$, positive for $x\gt 0$, negative for $x\lt 0$). Thus $(0,0)$ is a **Saddle Point**.

4. **Classify Point $(\frac{3}{2}, \frac{3}{2})$:**

$$
   H\left(\frac{3}{2}, \frac{3}{2}\right) = \begin{bmatrix} 9 & -9 \\ -9 & -9 + 12(\frac{9}{4}) \end{bmatrix} = \begin{bmatrix} 9 & -9 \\ -9 & 18 \end{bmatrix}
$$

$$
   \det(H) = (9)(18) - (-9)^2 = 162 - 81 = 81 \gt 0, \quad f_{xx} = 9 \gt 0 \implies \text{Strict Local Minimum}
$$

5. **Classify Point $(\frac{3}{2}, -\frac{3}{2})$:**

$$
   H\left(\frac{3}{2}, -\frac{3}{2}\right) = \begin{bmatrix} 9 & 9 \\ 9 & -9 + 12(\frac{9}{4}) \end{bmatrix} = \begin{bmatrix} 9 & 9 \\ 9 & 18 \end{bmatrix}
$$

$$
   \det(H) = (9)(18) - (9)^2 = 81 \gt 0, \quad f_{xx} = 9 \gt 0 \implies \text{Strict Local Minimum}
$$

$$
\boxed{\text{Critical points: } (0,0) \text{ [Saddle Point]}, \left(\frac{3}{2}, \frac{3}{2}\right) \text{ [Local Minimum]}, \left(\frac{3}{2}, -\frac{3}{2}\right) \text{ [Local Minimum]}}
$$

#### Key Insight / Takeaway
When $\det(H) = 0$, higher-order directional behavior must be examined. Non-degenerate critical points with $\det(H) \gt 0$ and $f_{xx} \gt 0$ guarantee strict local minima.

**Verification.** The cell below recomputes the boxed answer of Problem L1.2 and asserts agreement.

In [11]:
# Every critical point of f = x^3 - 3xy^2 + y^4 and its Hessian verdict.
x, y = sp.symbols("x y", real=True)
f = x**3 - 3 * x * y**2 + y**4
crit = sp.solve([sp.diff(f, x), sp.diff(f, y)], [x, y], dict=True)
print("critical points:", [(c[x], c[y]) for c in crit])
assert {(c[x], c[y]) for c in crit} == {(0, 0), (sp.Rational(3, 2), sp.Rational(3, 2)),
                                        (sp.Rational(3, 2), -sp.Rational(3, 2))}
Hs = sp.hessian(f, (x, y))
for c in sorted(crit, key=lambda d: (float(d[x]), float(d[y]))):
    Hc = Hs.subs(c)
    d, tr = sp.simplify(Hc.det()), sp.simplify(Hc.trace())
    verdict = ("inconclusive (det H = 0)" if d == 0 else
               "saddle" if d < 0 else ("local minimum" if Hc[0, 0] > 0 else "local maximum"))
    print(f"  ({c[x]}, {c[y]}): H = {Hc.tolist()}, det = {d}, trace = {tr}  ->  {verdict}")

# At the origin the test is silent, so probe the function directly along y = 0.
f_num = sp.lambdify((x, y), f, "numpy")
t = np.array([-1e-3, -1e-4, 1e-4, 1e-3])
print("\nf(t, 0) near the origin:", f_num(t, 0.0))
assert f_num(-1e-3, 0.0) < 0 < f_num(1e-3, 0.0)
print("f changes sign through (0,0), so it is not a local extremum there.")

critical points: [(0, 0), (3/2, -3/2), (3/2, 3/2)]
  (0, 0): H = [[0, 0], [0, 0]], det = 0, trace = 0  ->  inconclusive (det H = 0)
  (3/2, -3/2): H = [[9, 9], [9, 18]], det = 81, trace = 27  ->  local minimum
  (3/2, 3/2): H = [[9, -9], [-9, 18]], det = 81, trace = 27  ->  local minimum



f(t, 0) near the origin: [-0. -0.  0.  0.]
f changes sign through (0,0), so it is not a local extremum there.


---

### Problem L1.3 — Critical Points of $f(x,y) = e^{x^2 - y^2}(x^2 + 2y^2)$
**Source:** Demidovich No. 3208.  
**Problem Statement:**  
Find and classify all critical points of $f(x,y) = e^{x^2 - y^2}(x^2 + 2y^2)$.

#### First-Principles Intuition
Since $e^{x^2 - y^2} \gt 0$, critical points occur where the gradient vector vanishes. We factor common terms out of partial derivatives to solve for stationarity.

#### Step-by-Step Solution
1. **Compute Gradient:**

$$
   f_x = 2x e^{x^2 - y^2}(x^2 + 2y^2) + e^{x^2 - y^2}(2x) = 2x e^{x^2 - y^2}(x^2 + 2y^2 + 1) = 0
$$

$$
   f_y = -2y e^{x^2 - y^2}(x^2 + 2y^2) + e^{x^2 - y^2}(4y) = 2y e^{x^2 - y^2}(2 - x^2 - 2y^2) = 0
$$

2. **Solve $\nabla f = \mathbf{0}$:**
   Since $e^{x^2 - y^2} \gt 0$ and $(x^2 + 2y^2 + 1) \ge 1 \gt 0$:
   - From $f_x = 0 \implies x = 0$.
   - Substitute $x = 0$ into $f_y = 0$: $2y(2 - 2y^2) = 4y(1 - y^2) = 0 \implies y = 0$ or $y = \pm 1$.

   Critical points: $(0,0)$, $(0,1)$, and $(0,-1)$.

3. **Compute Hessian Matrix:**
   At the origin the second-derivative test already decides. Differentiating $f_x = 2x e^{x^2-y^2}(x^2+2y^2+1)$ in $x$ and $f_y = 2y e^{x^2-y^2}(2-x^2-2y^2)$ in $y$ and setting $x=y=0$ gives

$$
   H(0,0) = \begin{bmatrix} 2 & 0 \\ 0 & 4 \end{bmatrix} \succ 0 \implies (0,0) \text{ is a strict local minimum (Theorem 4.4).}
$$

   It is in fact **global**: $e^{x^2-y^2} \gt 0$ and $x^2 + 2y^2 \gt 0$ for $(x,y) \neq (0,0)$, so $f \gt 0 = f(0,0)$ everywhere else.

   To classify $(0, \pm 1)$, compute second partials at $(0, y)$ where $y = \pm 1$:

$$
   f_{xx}(0, \pm 1) = 2 e^{-1}(0 + 2(1)^2 + 1) = 6 e^{-1}
$$

$$
   f_{xy}(0, \pm 1) = 0 \quad (\text{due to } x=0 \text{ symmetry})
$$

$$
   f_{yy}(0, y) = \frac{\partial}{\partial y} [2y e^{-y^2}(2 - 2y^2)] = 2 e^{-y^2}(2 - 2y^2) + 2y (-2y) e^{-y^2}(2 - 2y^2) + 2y e^{-y^2}(-4y)
$$

   At $y = \pm 1$, $1 - y^2 = 0$:

$$
   f_{yy}(0, \pm 1) = 0 + 0 + 2(\pm 1) e^{-1} (-4(\pm 1)) = -8 e^{-1}
$$

   Hessian matrix at $(0, \pm 1)$:

$$
   H(0, \pm 1) = \begin{bmatrix} 6 e^{-1} & 0 \\ 0 & -8 e^{-1} \end{bmatrix} \implies \det(H) = -48 e^{-2} \lt 0
$$

$$
\boxed{\text{Critical points: } (0,0) \text{ [Strict Local/Global Minimum]}, (0, 1) \text{ [Saddle Point]}, (0, -1) \text{ [Saddle Point]}}
$$

#### Key Insight / Takeaway
Product rule differentiation often allows factoring out strictly positive exponential terms $e^{g(x)}$, simplifying critical point identification.

**Verification.** The cell below recomputes the boxed answer of Problem L1.3 and asserts agreement.

In [12]:
# Critical points of f = exp(x^2 - y^2)(x^2 + 2y^2) and their Hessians.
x, y = sp.symbols("x y", real=True)
f = sp.exp(x**2 - y**2) * (x**2 + 2 * y**2)
crit = sp.solve([sp.diff(f, x), sp.diff(f, y)], [x, y], dict=True)
print("critical points:", sorted((c[x], c[y]) for c in crit))
assert sorted((c[x], c[y]) for c in crit) == [(0, -1), (0, 0), (0, 1)]
Hs = sp.hessian(f, (x, y))
for pt in [(0, 0), (0, 1), (0, -1)]:
    Hc = sp.simplify(Hs.subs({x: pt[0], y: pt[1]}))
    d = sp.simplify(Hc.det())
    verdict = "saddle" if d < 0 else ("local minimum" if Hc[0, 0] > 0 else "local maximum")
    print(f"  {pt}: H = {Hc.tolist()}, det = {sp.nsimplify(d)} = {float(d):+.6f}  ->  {verdict}")
assert sp.simplify(Hs.subs({x: 0, y: 0}) - sp.Matrix([[2, 0], [0, 4]])) == sp.zeros(2, 2)
assert float(sp.simplify(Hs.subs({x: 0, y: 1}).det())) < 0

# The origin is a global minimum: f >= 0 everywhere with equality only there.
fn = sp.lambdify((x, y), f, "numpy")
P = rng.normal(size=(200000, 2)) * 2.0
print(f"\nmin over 200000 random points = {fn(P[:, 0], P[:, 1]).min():.6e}   f(0,0) = {fn(0.0, 0.0):.1f}")
assert fn(P[:, 0], P[:, 1]).min() > 0

critical points: [(0, -1), (0, 0), (0, 1)]
  (0, 0): H = [[2, 0], [0, 4]], det = 8 = +8.000000  ->  local minimum
  (0, 1): H = [[6*exp(-1), 0], [0, -8*exp(-1)]], det = -48*exp(-2) = -6.496094  ->  saddle
  (0, -1): H = [[6*exp(-1), 0], [0, -8*exp(-1)]], det = -48*exp(-2) = -6.496094  ->  saddle

min over 200000 random points = 4.783128e-34   f(0,0) = 0.0


---

### Problem L1.4 — Multivariable 2nd-Order Taylor Polynomial of $f(x,y) = \cos(x + 2y)$ at $(0,0)$
**Source:** Adapted from Apostol, *Mathematical Analysis* (Ch. 12).  
**Problem Statement:**  
Find the 2nd-order Taylor polynomial $T_2(x,y)$ of $f(x,y) = \cos(x + 2y)$ around $(0,0)$ using partial derivatives and the Hessian matrix.

#### First-Principles Intuition
The 2nd-order Taylor formula is:

$$
T_2(x,y) = f(0,0) + \nabla f(0,0)^T \begin{bmatrix} x \\ y \end{bmatrix} + \frac{1}{2} \begin{bmatrix} x & y \end{bmatrix} H(0,0) \begin{bmatrix} x \\ y \end{bmatrix}
$$

#### Step-by-Step Solution
1. **Evaluate $f(0,0)$:**

$$
   f(0,0) = \cos(0) = 1
$$

2. **Compute Gradient at $(0,0)$:**

$$
   f_x = -\sin(x + 2y) \implies f_x(0,0) = 0
$$

$$
   f_y = -2\sin(x + 2y) \implies f_y(0,0) = 0
$$

   Thus $\nabla f(0,0) = \begin{bmatrix} 0 \\ 0 \end{bmatrix}$.

3. **Compute Hessian at $(0,0)$:**

$$
   f_{xx} = -\cos(x + 2y) \implies f_{xx}(0,0) = -1
$$

$$
   f_{xy} = -2\cos(x + 2y) \implies f_{xy}(0,0) = -2
$$

$$
   f_{yy} = -4\cos(x + 2y) \implies f_{yy}(0,0) = -4
$$

$$
   H(0,0) = \begin{bmatrix} -1 & -2 \\ -2 & -4 \end{bmatrix}
$$

4. **Construct $T_2(x,y)$:**

$$
   T_2(x,y) = 1 + 0 + \frac{1}{2} \begin{bmatrix} x & y \end{bmatrix} \begin{bmatrix} -1 & -2 \\ -2 & -4 \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} = 1 + \frac{1}{2} (-x^2 - 4xy - 4y^2) = 1 - \frac{1}{2}(x + 2y)^2
$$

$$
\boxed{T_2(x,y) = 1 - \frac{1}{2} x^2 - 2xy - 2y^2 = 1 - \frac{1}{2}(x + 2y)^2}
$$

#### Key Insight / Takeaway
Using the single-variable expansion $\cos(u) \approx 1 - \frac{1}{2} u^2$ with $u = x + 2y$ yields $1 - \frac{1}{2}(x + 2y)^2$ directly, matching the explicit multivariable Hessian matrix construction.

**Verification.** The cell below recomputes the boxed answer of Problem L1.4 and asserts agreement.

In [13]:
# T2 for cos(x + 2y) at the origin, from the Hessian and from the one-variable series.
x, y = sp.symbols("x y", real=True)
f = sp.cos(x + 2 * y)
H0 = sp.hessian(f, (x, y)).subs({x: 0, y: 0})
print("H(0,0) =", H0.tolist())
assert H0 == sp.Matrix([[-1, -2], [-2, -4]])
h = sp.Matrix([x, y])
T2 = sp.expand(f.subs({x: 0, y: 0}) + sp.Matrix([sp.diff(f, x), sp.diff(f, y)]).subs({x: 0, y: 0}).dot(h)
               + sp.Rational(1, 2) * (h.T * H0 * h)[0, 0])
print("T2(x,y)  =", T2)
assert sp.simplify(T2 - (1 - sp.Rational(1, 2) * (x + 2 * y) ** 2)) == 0

# Cubic-order agreement: the error must fall like ||h||^4 here, since the h^3 term vanishes.
fn, Tn = sp.lambdify((x, y), f, "numpy"), sp.lambdify((x, y), T2, "numpy")
for s in (1e-1, 1e-2, 1e-3):
    print(f"  ||h|| ~ {s:.0e}:  |f - T2| = {abs(fn(s, s) - Tn(s, s)):.3e}")
assert abs(fn(1e-2, 1e-2) - Tn(1e-2, 1e-2)) < 1e-6

H(0,0) = [[-1, -2], [-2, -4]]
T2(x,y)  = -x**2/2 - 2*x*y - 2*y**2 + 1
  ||h|| ~ 1e-01:  |f - T2| = 3.365e-04
  ||h|| ~ 1e-02:  |f - T2| = 3.375e-08
  ||h|| ~ 1e-03:  |f - T2| = 3.375e-12


---

### Problem L1.5 — Rayleigh Quotient Bounds for a Symmetric Matrix
**Source:** Adapted from Horn & Johnson, *Matrix Analysis* (Ch. 4).  
**Problem Statement:**  
Let:

$$
A = \begin{bmatrix} 4 & 1 & 0 \\ 1 & 4 & 0 \\ 0 & 0 & 2 \end{bmatrix}
$$

Compute all eigenvalues of $A$, and determine the exact minimum and maximum possible values of the Rayleigh quotient $R_A(x) = \frac{x^T A x}{x^T x}$ for $x \neq \mathbf{0}$.

#### First-Principles Intuition
By the Rayleigh-Ritz Theorem, the range of the Rayleigh quotient $R_A(x)$ over non-zero vectors is the closed interval $[\lambda_{\min}, \lambda_{\max}]$.

#### Step-by-Step Solution
Compute characteristic equation $\det(A - \lambda I) = 0$:

$$
\det \begin{bmatrix} 4 - \lambda & 1 & 0 \\ 1 & 4 - \lambda & 0 \\ 0 & 0 & 2 - \lambda \end{bmatrix} = (2 - \lambda) \left[ (4 - \lambda)^2 - 1 \right] = 0
$$

Solve $(4 - \lambda)^2 = 1 \implies 4 - \lambda = \pm 1 \implies \lambda = 3 \text{ or } \lambda = 5$.  
The three eigenvalues are:

$$
\lambda_1 = 2, \quad \lambda_2 = 3, \quad \lambda_3 = 5
$$

Identify minimum and maximum eigenvalues:

$$
\lambda_{\min} = 2, \quad \lambda_{\max} = 5
$$

By Theorem 4.3 (Rayleigh–Ritz) of [`first_principles.ipynb`](first_principles.ipynb):

$$
2 \le \frac{x^T A x}{x^T x} \le 5
$$

$$
\boxed{\min_{x \neq \mathbf{0}} R_A(x) = 2, \quad \max_{x \neq \mathbf{0}} R_A(x) = 5}
$$

#### Key Insight / Takeaway
The Rayleigh quotient bounds are determined solely by the extreme eigenvalues $\lambda_{\min}$ and $\lambda_{\max}$, irrespective of intermediate spectrum values.

**Verification.** The cell below recomputes the boxed answer of Problem L1.5 and asserts agreement.

In [14]:
# Eigenvalues of A and the exact range of its Rayleigh quotient.
A = np.array([[4.0, 1.0, 0.0], [1.0, 4.0, 0.0], [0.0, 0.0, 2.0]])
lam, U = np.linalg.eigh(A)
print("eigenvalues:", lam)
report("L1.5  spectrum", np.sort(lam), np.array([2.0, 3.0, 5.0]))
R = lambda v: float(v @ A @ v / (v @ v))
report("L1.5  R at the lambda_min eigenvector", R(U[:, 0]), 2.0)
report("L1.5  R at the lambda_max eigenvector", R(U[:, -1]), 5.0)
vals = np.array([R(v) for v in rng.normal(size=(200000, 3))])
print(f"\n200000 random directions: min R = {vals.min():.6f}, max R = {vals.max():.6f}")
assert 2.0 - 1e-12 <= vals.min() and vals.max() <= 5.0 + 1e-12

eigenvalues: [2. 3. 5.]
L1.5  spectrum                                       residual = 0.000e+00
L1.5  R at the lambda_min eigenvector                residual = 0.000e+00
L1.5  R at the lambda_max eigenvector                residual = 0.000e+00



200000 random directions: min R = 2.000020, max R = 4.999993


---

### Problem L1.6 — Explicit Step of 2D Newton's Method
**Source:** Adapted from Nocedal & Wright, *Numerical Optimization* (Ch. 3).  
**Problem Statement:**  
Consider the quadratic function $f(x,y) = 2x^2 + y^2 - 2xy + 3x$. Compute the gradient $\nabla f(x,y)$, the Hessian matrix $H$, and perform one step of multivariable Newton's method starting from initial point $(x_0, y_0) = (0,0)^T$.

#### First-Principles Intuition
Newton's method updates $x_1 = x_0 - H^{-1} \nabla f(x_0)$. For a strictly convex quadratic function, Newton's method reaches the exact minimizer in a single step.

#### Step-by-Step Solution
1. **Compute Gradient:**

$$
   \nabla f(x,y) = \begin{bmatrix} 4x - 2y + 3 \\ 2y - 2x \end{bmatrix}
$$

   At initial point $(0,0)^T$:

$$
   \nabla f(0,0) = \begin{bmatrix} 3 \\ 0 \end{bmatrix}
$$

2. **Compute Hessian Matrix:**

$$
   H = \begin{bmatrix} f_{xx} & f_{xy} \\ f_{yx} & f_{yy} \end{bmatrix} = \begin{bmatrix} 4 & -2 \\ -2 & 2 \end{bmatrix}
$$

3. **Invert $H$:**

$$
   \det(H) = (4)(2) - (-2)(-2) = 8 - 4 = 4
$$

$$
   H^{-1} = \frac{1}{4} \begin{bmatrix} 2 & 2 \\ 2 & 4 \end{bmatrix} = \begin{bmatrix} \frac{1}{2} & \frac{1}{2} \\ \frac{1}{2} & 1 \end{bmatrix}
$$

4. **Compute Newton Step:**

$$
   \begin{bmatrix} x_1 \\ y_1 \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \end{bmatrix} - H^{-1} \nabla f(0,0) = -\begin{bmatrix} \frac{1}{2} & \frac{1}{2} \\ \frac{1}{2} & 1 \end{bmatrix} \begin{bmatrix} 3 \\ 0 \end{bmatrix} = -\begin{bmatrix} \frac{3}{2} \\ \frac{3}{2} \end{bmatrix} = \begin{bmatrix} -\frac{3}{2} \\ -\frac{3}{2} \end{bmatrix}
$$

5. **Verify Stationarity at $( -\frac{3}{2}, -\frac{3}{2} )$:**

$$
   \nabla f(-\frac{3}{2}, -\frac{3}{2}) = \begin{bmatrix} 4(-\frac{3}{2}) - 2(-\frac{3}{2}) + 3 \\ 2(-\frac{3}{2}) - 2(-\frac{3}{2}) \end{bmatrix} = \begin{bmatrix} -6 + 3 + 3 \\ -3 + 3 \end{bmatrix} = \begin{bmatrix} 0 \\ 0 \end{bmatrix}
$$

$$
\boxed{\left(x_1, y_1\right) = \left(-\frac{3}{2}, -\frac{3}{2}\right) \quad (\text{exact global minimizer}) }
$$

#### Key Insight / Takeaway
Newton's method solves quadratic minimization problems in exactly one iteration because the second-order Taylor model is exact.

**Verification.** The cell below recomputes the boxed answer of Problem L1.6 and asserts agreement.

In [15]:
# One Newton step on f = 2x^2 + y^2 - 2xy + 3x from the origin.
f = lambda v: 2 * v[0]**2 + v[1]**2 - 2 * v[0] * v[1] + 3 * v[0]
grad = lambda v: np.array([4 * v[0] - 2 * v[1] + 3, 2 * v[1] - 2 * v[0]])
H = np.array([[4.0, -2.0], [-2.0, 2.0]])
print(f"det H = {np.linalg.det(H):.6f}   eigenvalues = {np.linalg.eigvalsh(H)}")
x0 = np.zeros(2)
x1 = x0 - np.linalg.solve(H, grad(x0))
print("Newton iterate x1 =", x1)
report("L1.6  x1", x1, np.array([-1.5, -1.5]))
report("L1.6  gradient at x1", grad(x1), np.zeros(2))
best = min(f(x1 + 1e-3 * d) for d in rng.normal(size=(20000, 2)))
print(f"\nf(x1) = {f(x1):.10f}   best nearby value = {best:.10f}")
assert f(x1) < best

det H = 4.000000   eigenvalues = [0.7639 5.2361]
Newton iterate x1 = [-1.5 -1.5]
L1.6  x1                                             residual = 0.000e+00
L1.6  gradient at x1                                 residual = 0.000e+00

f(x1) = -2.2500000000   best nearby value = -2.2499999999


---

### Problem L1.7 — Sylvester's Criterion for a $3 \times 3$ Hessian Matrix
**Source:** Adapted from Horn & Johnson, *Matrix Analysis* (Ch. 7).  
**Problem Statement:**  
Determine whether the symmetric matrix:

$$
H = \begin{bmatrix} 2 & -1 & 0 \\ -1 & 2 & -1 \\ 0 & -1 & 2 \end{bmatrix}
$$

is positive definite using Sylvester's Criterion (leading principal minors).

#### First-Principles Intuition
Sylvester's criterion states that a real symmetric matrix is positive definite if and only if all upper-left square sub-determinants ($\Delta_1, \Delta_2, \dots, \Delta_n$) are strictly positive.

#### Step-by-Step Solution
1. **First Leading Principal Minor ($\Delta_1$):**

$$
   \Delta_1 = \lvert 2 \rvert = 2 \gt 0
$$

2. **Second Leading Principal Minor ($\Delta_2$):**

$$
   \Delta_2 = \begin{vmatrix} 2 & -1 \\ -1 & 2 \end{vmatrix} = (2)(2) - (-1)(-1) = 4 - 1 = 3 \gt 0
$$

3. **Third Leading Principal Minor ($\Delta_3 = \det(H)$):**
   Expand along the first row:

$$
   \Delta_3 = 2 \begin{vmatrix} 2 & -1 \\ -1 & 2 \end{vmatrix} - (-1) \begin{vmatrix} -1 & -1 \\ 0 & 2 \end{vmatrix} + 0 = 2(3) + 1(-2 - 0) = 6 - 2 = 4 \gt 0
$$

Since $\Delta_1 = 2 \gt 0$, $\Delta_2 = 3 \gt 0$, and $\Delta_3 = 4 \gt 0$, all leading principal minors are strictly positive.

$$
\boxed{\Delta_1 = 2 \gt 0, \ \Delta_2 = 3 \gt 0, \ \Delta_3 = 4 \gt 0 \implies H \text{ is strictly positive definite } (H \succ 0)}
$$

#### Key Insight / Takeaway
Sylvester's Criterion provides a determinant-based verification of positive definiteness without computing eigenvalues.

**Verification.** The cell below recomputes the boxed answer of Problem L1.7 and asserts agreement.

In [16]:
# Sylvester's criterion on the 1D-Laplacian block, cross-checked against the spectrum.
H = np.array([[2.0, -1.0, 0.0], [-1.0, 2.0, -1.0], [0.0, -1.0, 2.0]])
minors = [np.linalg.det(H[:k, :k]) for k in (1, 2, 3)]
print("leading principal minors:", [f"{m:.6f}" for m in minors])
report("L1.7  minors", minors, [2.0, 3.0, 4.0], tol=1e-12)
lam = np.linalg.eigvalsh(H)
print("eigenvalues:", lam)
assert all(m > 0 for m in minors) and lam.min() > 0

# Non-leading minors are not enough: diag(1, -1) has a positive minor and is indefinite.
C = np.diag([1.0, -1.0])
print(f"\ncounterexample diag(1,-1): non-leading minor C[1,1] = {C[1,1]:+.1f}, "
      f"leading minors = {[np.linalg.det(C[:k,:k]) for k in (1,2)]}, eigenvalues = {np.linalg.eigvalsh(C)}")
assert np.linalg.eigvalsh(C).min() < 0

leading principal minors: ['2.000000', '3.000000', '4.000000']
L1.7  minors                                         residual = 4.441e-16
eigenvalues: [0.5858 2.     3.4142]

counterexample diag(1,-1): non-leading minor C[1,1] = -1.0, leading minors = [np.float64(1.0), np.float64(-1.0)], eigenvalues = [-1.  1.]


---

### Problem L1.8 — Jacobian Matrix of a Neural Network Linear Layer
**Source:** Adapted from Goodfellow et al., *Deep Learning* (Ch. 6).  
**Problem Statement:**  
Let $y = f(x) = W x + b$, where $x \in \mathbb{R}^n$, $W \in \mathbb{R}^{m \times n}$, $b \in \mathbb{R}^m$, and $y \in \mathbb{R}^m$. Derive the Jacobian matrix $\frac{\partial y}{\partial x} \in \mathbb{R}^{m \times n}$ and the Jacobian matrix with respect to parameters $\frac{\partial y}{\partial b} \in \mathbb{R}^{m \times m}$.

#### First-Principles Intuition
A linear layer applies a matrix transformation and translation. Derivatives with respect to inputs yield the weight matrix $W$, while derivatives with respect to bias yield the identity matrix $I$.

#### Step-by-Step Solution
1. **Jacobian with respect to $x$ ($\frac{\partial y}{\partial x}$):**
   The $i$-th element of $y$ is $y_i = \sum_{k=1}^n W_{ik} x_k + b_i$.
   Differentiating with respect to $x_j$:

$$
   \left( \frac{\partial y}{\partial x} \right)_{ij} = \frac{\partial y_i}{\partial x_j} = W_{ij} \implies \boxed{\frac{\partial y}{\partial x} = W}
$$

2. **Jacobian with respect to $b$ ($\frac{\partial y}{\partial b}$):**
   Differentiating $y_i$ with respect to $b_j$:

$$
   \left( \frac{\partial y}{\partial b} \right)_{ij} = \frac{\partial y_i}{\partial b_j} = \delta_{ij} \implies \boxed{\frac{\partial y}{\partial b} = I_m}
$$

#### Key Insight / Takeaway
In neural network backpropagation, vector-matrix layer derivatives yield linear weight transformations for error propagation.

**Verification.** The cell below recomputes the boxed answer of Problem L1.8 and asserts agreement.

In [17]:
# dy/dx = W and dy/db = I for the affine layer y = Wx + b.
m, n = 4, 6
W = rng.normal(size=(m, n))
b = rng.normal(size=m)
x0 = rng.normal(size=n)

def jac_fd(fun, v, h=1e-6):
    out = fun(v)
    J = np.zeros((out.size, v.size))
    for j in range(v.size):
        e = np.zeros(v.size); e[j] = h
        J[:, j] = (fun(v + e) - fun(v - e)) / (2 * h)
    return J

report("L1.8  dy/dx vs W", jac_fd(lambda v: W @ v + b, x0), W, tol=1e-8)
report("L1.8  dy/db vs I_m", jac_fd(lambda v: W @ x0 + v, b), np.eye(m), tol=1e-8)
print("\nBoth Jacobians are constant: the layer is affine in x and in b separately.")

L1.8  dy/dx vs W                                     residual = 6.716e-10
L1.8  dy/db vs I_m                                   residual = 3.043e-10

Both Jacobians are constant: the layer is affine in x and in b separately.


---

### Problem L1.9 — Gradient and Hessian of Quadratic Function $f(x) = \frac{1}{2} x^T A x - b^T x + c$
**Source:** Adapted from Boyd & Vandenberghe, *Convex Optimization* (Ch. 2).  
**Problem Statement:**  
Let $A \in \mathbb{R}^{n \times n}$ be a symmetric matrix, $b \in \mathbb{R}^n$, and $c \in \mathbb{R}$. Derive the gradient $\nabla f(x)$ and Hessian matrix $H_f(x)$ of $f(x) = \frac{1}{2} x^T A x - b^T x + c$.

#### First-Principles Intuition
Quadratic forms in vector calculus parallel $f(x) = \frac{1}{2} a x^2 - b x + c$ in single-variable calculus, where the first derivative is $a x - b$ and the second derivative is $a$.

#### Step-by-Step Solution
Express $f(x)$ in index summation notation:

$$
f(x) = \frac{1}{2} \sum_{i=1}^n \sum_{j=1}^n A_{ij} x_i x_j - \sum_{i=1}^n b_i x_i + c
$$

Compute the partial derivative with respect to $x_k$:

$$
\frac{\partial f}{\partial x_k} = \frac{1}{2} \sum_{j=1}^n A_{kj} x_j + \frac{1}{2} \sum_{i=1}^n A_{ik} x_i - b_k
$$

Since $A$ is symmetric ($A_{ik} = A_{ki}$):

$$
\frac{\partial f}{\partial x_k} = \sum_{j=1}^n A_{kj} x_j - b_k = (A x)_k - b_k \implies \nabla f(x) = A x - b
$$

Compute second partial derivatives:

$$
\frac{\partial^2 f}{\partial x_k \partial x_l} = \frac{\partial}{\partial x_l} \left( \sum_{j=1}^n A_{kj} x_j - b_k \right) = A_{kl}
$$

Thus, the Hessian matrix is constant and equal to $A$:

$$
\boxed{\nabla f(x) = A x - b, \quad H_f(x) = A}
$$

#### Key Insight / Takeaway
The matrix $A$ in a quadratic form $\frac{1}{2}x^T A x$ acts as the constant multidimensional second derivative.

**Verification.** The cell below recomputes the boxed answer of Problem L1.9 and asserts agreement.

In [18]:
# grad f = Ax - b and Hessian = A for the quadratic, against finite differences.
n = 5
B = rng.normal(size=(n, n))
A = 0.5 * (B + B.T)                      # symmetric, as the problem assumes
b = rng.normal(size=n)
c = 0.7
f = lambda v: 0.5 * v @ A @ v - b @ v + c
x0 = rng.normal(size=n)

h = 1e-5
g_fd = np.array([(f(x0 + h * np.eye(n)[j]) - f(x0 - h * np.eye(n)[j])) / (2 * h) for j in range(n)])
report("L1.9  finite-difference gradient vs Ax - b", g_fd, A @ x0 - b, tol=1e-6)

hh = 1e-4
H_fd = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        ei, ej = hh * np.eye(n)[i], hh * np.eye(n)[j]
        H_fd[i, j] = (f(x0 + ei + ej) - f(x0 + ei - ej) - f(x0 - ei + ej) + f(x0 - ei - ej)) / (4 * hh**2)
report("L1.9  finite-difference Hessian vs A", H_fd, A, tol=1e-5)
print("\nThe Hessian does not depend on x: it is the constant matrix A.")

# Without symmetry only the symmetric part survives in the Hessian.
Bn = rng.normal(size=(n, n))
fn = lambda v: 0.5 * v @ Bn @ v
H_fd2 = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        ei, ej = hh * np.eye(n)[i], hh * np.eye(n)[j]
        H_fd2[i, j] = (fn(x0 + ei + ej) - fn(x0 + ei - ej) - fn(x0 - ei + ej) + fn(x0 - ei - ej)) / (4 * hh**2)
report("L1.9  non-symmetric B: Hessian vs (B + B^T)/2", H_fd2, 0.5 * (Bn + Bn.T), tol=1e-5)

L1.9  finite-difference gradient vs Ax - b           residual = 3.780e-11
L1.9  finite-difference Hessian vs A                 residual = 2.082e-08

The Hessian does not depend on x: it is the constant matrix A.
L1.9  non-symmetric B: Hessian vs (B + B^T)/2        residual = 2.671e-08


---

### Problem L1.10 — Jacobian Matrix & Metric Tensor for Cylindrical Coordinates
**Source:** Adapted from Spivak, *Calculus on Manifolds*.  
**Problem Statement:**  
The transformation from cylindrical coordinates $(r, \theta, z)$ to Cartesian coordinates $(x, y, z)$ is $x = r \cos \theta$, $y = r \sin \theta$, $z = z$. Compute the Jacobian matrix $J$ and the Riemannian metric tensor $g = J^T J$.

#### First-Principles Intuition
The metric tensor $g = J^T J$ measures squared differential arc length $ds^2 = dx^2 + dy^2 + dz^2$ in curvilinear coordinates: $ds^2 = \sum g_{ij} dq^i dq^j$.

#### Step-by-Step Solution
1. **Compute Jacobian Matrix $J$:**

$$
   J = \begin{bmatrix}
   \frac{\partial x}{\partial r} & \frac{\partial x}{\partial \theta} & \frac{\partial x}{\partial z} \\[4pt]
   \frac{\partial y}{\partial r} & \frac{\partial y}{\partial \theta} & \frac{\partial y}{\partial z} \\[4pt]
   \frac{\partial z}{\partial r} & \frac{\partial z}{\partial \theta} & \frac{\partial z}{\partial z}
   \end{bmatrix} = \begin{bmatrix}
   \cos \theta & -r \sin \theta & 0 \\
   \sin \theta & r \cos \theta & 0 \\
   0 & 0 & 1
   \end{bmatrix}
$$

2. **Compute Metric Tensor $g = J^T J$:**

$$
   J^T J = \begin{bmatrix}
   \cos \theta & \sin \theta & 0 \\
   -r \sin \theta & r \cos \theta & 0 \\
   0 & 0 & 1
   \end{bmatrix} \begin{bmatrix}
   \cos \theta & -r \sin \theta & 0 \\
   \sin \theta & r \cos \theta & 0 \\
   0 & 0 & 1
   \end{bmatrix}
$$

   - Row 1 $\cdot$ Col 1: $\cos^2 \theta + \sin^2 \theta = 1$
   - Row 1 $\cdot$ Col 2: $-r \cos \theta \sin \theta + r \sin \theta \cos \theta = 0$
   - Row 2 $\cdot$ Col 2: $r^2 \sin^2 \theta + r^2 \cos^2 \theta = r^2$
   - Row 3 $\cdot$ Col 3: $1$

   All off-diagonal entries vanish, giving:

$$
\boxed{J = \begin{bmatrix} \cos \theta & -r \sin \theta & 0 \\ \sin \theta & r \cos \theta & 0 \\ 0 & 0 & 1 \end{bmatrix}, \quad g = J^T J = \begin{bmatrix} 1 & 0 & 0 \\ 0 & r^2 & 0 \\ 0 & 0 & 1 \end{bmatrix}}
$$

#### Key Insight / Takeaway
The metric tensor $g = J^T J$ is diagonal because cylindrical coordinate basis vectors are orthogonal, yielding line element $ds^2 = dr^2 + r^2 d\theta^2 + dz^2$.

**Verification.** The cell below recomputes the boxed answer of Problem L1.10 and asserts agreement.

In [19]:
# Cylindrical Jacobian and metric tensor g = J^T J = diag(1, r^2, 1).
r_s, th_s, z_s = sp.symbols("r theta z", real=True)
Xc = sp.Matrix([r_s * sp.cos(th_s), r_s * sp.sin(th_s), z_s])
Jc = Xc.jacobian([r_s, th_s, z_s])
g = sp.simplify(Jc.T * Jc)
print("J =", Jc.tolist())
print("g = J^T J =", g.tolist())
assert sp.simplify(g - sp.diag(1, r_s**2, 1)) == sp.zeros(3, 3)
assert sp.simplify(Jc.det() - r_s) == 0
print("\ndet J = ", sp.simplify(Jc.det()), " and sqrt(det g) =", sp.simplify(sp.sqrt(g.det())))

# ds^2 = dr^2 + r^2 dtheta^2 + dz^2 : compare a chord length with the metric prediction.
r0, t0, z0, d = 1.4, 0.3, -0.2, 1e-5
p = lambda r, t, zz: np.array([r * np.cos(t), r * np.sin(t), zz])
chord = np.linalg.norm(p(r0 + d, t0 + d, z0 + d) - p(r0, t0, z0))
pred = np.sqrt(d**2 + r0**2 * d**2 + d**2)
print(f"chord = {chord:.12e}   sqrt(dq^T g dq) = {pred:.12e}")
assert abs(chord - pred) / pred < 1e-5

J = [[cos(theta), -r*sin(theta), 0], [sin(theta), r*cos(theta), 0], [0, 0, 1]]
g = J^T J = [[1, 0, 0], [0, r**2, 0], [0, 0, 1]]

det J =  r  and sqrt(det g) = Abs(r)


chord = 1.989978391846e-05   sqrt(dq^T g dq) = 1.989974874213e-05


---

## L2 — Applications (AI/ML and Physics)

### Problem L2.1 — Analytical Proof of Softmax Jacobian Properties
**Source:** Goodfellow et al., *Deep Learning* (Ch. 4).  
**Problem Statement:**  
Let $S(z) \in \mathbb{R}^n$ be the Softmax function $S_i(z) = \frac{e^{z_i}}{\sum_k e^{z_k}}$. Derive $J_{ij} = \frac{\partial S_i}{\partial z_j} = S_i (\delta_{ij} - S_j)$ and prove that $\sum_{j=1}^n J_{ij} = 0$ for every row $i$.

#### First-Principles Intuition
The row sum $\sum_j \frac{\partial S_i}{\partial z_j}$ represents the directional derivative of $S_i$ when all logits increase simultaneously by $\epsilon$. Since Softmax is invariant to uniform logit shifts, this row sum must equal 0.

#### Step-by-Step Solution
1. **Derivation of $J_{ij}$:**
   Let $K = \sum_{k=1}^n e^{z_k}$.
   - If $i = j$:

$$
     \frac{\partial S_i}{\partial z_i} = \frac{e^{z_i} K - e^{z_i} e^{z_i}}{K^2} = \frac{e^{z_i}}{K} \left( 1 - \frac{e^{z_i}}{K} \right) = S_i (1 - S_i)
$$

   - If $i \neq j$:

$$
     \frac{\partial S_i}{\partial z_j} = \frac{0 - e^{z_i} e^{z_j}}{K^2} = -S_i S_j
$$

   Combining both using Kronecker delta $\delta_{ij}$:

$$
   J_{ij} = S_i \delta_{ij} - S_i S_j = S_i (\delta_{ij} - S_j)
$$

2. **Compute Row Sum:**

$$
   \sum_{j=1}^n J_{ij} = \sum_{j=1}^n S_i (\delta_{ij} - S_j) = S_i \sum_{j=1}^n \delta_{ij} - S_i \sum_{j=1}^n S_j
$$

   Since $\sum_{j=1}^n \delta_{ij} = 1$ and $\sum_{j=1}^n S_j = 1$:

$$
   \sum_{j=1}^n J_{ij} = S_i(1) - S_i(1) = 0
$$

$$
\boxed{J_{ij} = S_i(\delta_{ij} - S_j) \implies J = \operatorname{diag}(S) - S S^T, \quad \sum_{j=1}^n J_{ij} = 0}
$$

#### Key Insight / Takeaway
Every row of the Softmax Jacobian sums to zero, confirming that constant offsets in input logits produce zero change in output probabilities.

**Verification.** The cell below recomputes the boxed answer of Problem L2.1 and asserts agreement.

In [20]:
# Softmax Jacobian: entrywise formula, matrix form, and zero row sums.
n = 7
z = rng.normal(size=n) * 2.0
S = softmax(z)

def jac_fd(fun, v, h=1e-6):
    out = fun(v)
    J = np.zeros((out.size, v.size))
    for j in range(v.size):
        e = np.zeros(v.size); e[j] = h
        J[:, j] = (fun(v + e) - fun(v - e)) / (2 * h)
    return J

J_formula = np.array([[S[i] * ((i == j) - S[j]) for j in range(n)] for i in range(n)])
J_matrix = np.diag(S) - np.outer(S, S)
report("L2.1  S_i(delta_ij - S_j) vs diag(S) - S S^T", J_formula, J_matrix, tol=1e-15)
report("L2.1  analytic J vs finite differences", J_matrix, jac_fd(softmax, z), tol=1e-8)
report("L2.1  row sums of J", J_matrix.sum(axis=1), np.zeros(n), tol=1e-15)
print("\nrow sums:", J_matrix.sum(axis=1))
c = 3.5
report("L2.1  shift invariance S(z + c1) - S(z)", softmax(z + c), S, tol=1e-14)

L2.1  S_i(delta_ij - S_j) vs diag(S) - S S^T         residual = 2.168e-19
L2.1  analytic J vs finite differences               residual = 4.737e-11
L2.1  row sums of J                                  residual = 2.949e-17

row sums: [0. 0. 0. 0. 0. 0. 0.]
L2.1  shift invariance S(z + c1) - S(z)              residual = 1.665e-16


---

### Problem L2.2 — Hessian Matrix of Softmax Cross-Entropy Loss
**Source:** Boyd & Vandenberghe, *Convex Optimization* (Ch. 3).  
**Problem Statement:**  
Let $\mathcal{L}(z) = -\ln S_y(z)$ be the multi-class cross-entropy loss for target class $y$, where $S_y(z)$ is the Softmax probability. Prove that $\nabla_z^2 \mathcal{L}(z) = \operatorname{diag}(S) - S S^T$, and determine its rank for an $n$-dimensional logit vector.

#### First-Principles Intuition
The cross-entropy loss gradient is $\nabla \mathcal{L} = S - e_y$. Differentiating this gradient with respect to $z$ yields the Jacobian of Softmax, which serves as the loss Hessian.

#### Step-by-Step Solution
1. **First Derivative (Gradient):**

$$
   \mathcal{L}(z) = -\ln \left( \frac{e^{z_y}}{\sum_k e^{z_k}} \right) = -z_y + \ln \left( \sum_{k=1}^n e^{z_k} \right)
$$

$$
   \frac{\partial \mathcal{L}}{\partial z_i} = -\delta_{yi} + \frac{e^{z_i}}{\sum_k e^{z_k}} = S_i - \delta_{yi} \implies \nabla_z \mathcal{L} = S - e_y
$$

2. **Second Derivative (Hessian Matrix):**
   Differentiate $\frac{\partial \mathcal{L}}{\partial z_i} = S_i - \delta_{yi}$ with respect to $z_j$:

$$
   H_{ij} = \frac{\partial^2 \mathcal{L}}{\partial z_i \partial z_j} = \frac{\partial S_i}{\partial z_j} - 0 = S_i \delta_{ij} - S_i S_j
$$

   Matrix form:

$$
   H = \operatorname{diag}(S) - S S^T
$$

3. **Determine Rank:**
   First, $H \mathbf{1} = \operatorname{diag}(S)\mathbf{1} - S(S^\top \mathbf{1}) = S - S = \mathbf{0}$, so $\operatorname{span}\lbrace\mathbf{1}\rbrace \subseteq \operatorname{Null}(H)$.

   For the reverse inclusion, use the variance identity of Problem L0.8 rather than orthogonality to $\mathbf{1}$ — a vector orthogonal to $\mathbf{1}$ satisfies $\sum_i v_i = 0$, which is **not** the same as the $S$-weighted centring $\sum_i S_i v_i = 0$, so orthogonality alone proves nothing here. Instead, for any $v$,

$$
   v^\top H v = \sum_{i=1}^n S_i v_i^2 - \Big( \sum_{i=1}^n S_i v_i \Big)^2 = \operatorname{Var}_{i \sim S}(v_i) .
$$

   A variance vanishes exactly when the random variable is almost surely constant. Since every $S_i \gt 0$, every index carries positive mass, so $\operatorname{Var}_{i \sim S}(v_i) = 0$ forces $v_i = c$ for all $i$, i.e. $v = c\mathbf{1}$. Because $H \succeq 0$, $Hv = \mathbf{0}$ implies $v^\top H v = 0$, hence $\operatorname{Null}(H) = \operatorname{span}\lbrace\mathbf{1}\rbrace$ exactly, a one-dimensional space. By the rank-nullity theorem:

$$
\boxed{H = \operatorname{diag}(S) - S S^T, \quad \operatorname{rank}(H) = n - 1}
$$

#### Key Insight / Takeaway
The Softmax Cross-Entropy loss Hessian is positive semi-definite with rank $n-1$, reflecting convexity with a 1D flat line of invariance along $\mathbf{1}$.

**Verification.** The cell below recomputes the boxed answer of Problem L2.2 and asserts agreement.

In [21]:
# Cross-entropy in logit space: gradient S - e_y, Hessian diag(S) - S S^T, rank n - 1.
n, y = 6, 2
z = rng.normal(size=n) * 1.5
loss = lambda v: -(v[y] - np.log(np.exp(v - v.max()).sum()) - v.max())
S = softmax(z)

h = 1e-6
g_fd = np.array([(loss(z + h * np.eye(n)[j]) - loss(z - h * np.eye(n)[j])) / (2 * h) for j in range(n)])
report("L2.2  gradient vs S - e_y", g_fd, S - np.eye(n)[y], tol=1e-7)

hh = 1e-4
H_fd = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        ei, ej = hh * np.eye(n)[i], hh * np.eye(n)[j]
        H_fd[i, j] = (loss(z + ei + ej) - loss(z + ei - ej) - loss(z - ei + ej) + loss(z - ei - ej)) / (4 * hh**2)
H = np.diag(S) - np.outer(S, S)
report("L2.2  Hessian vs diag(S) - S S^T", H_fd, H, tol=1e-5)

lam = np.linalg.eigvalsh(H)
print("\neigenvalues of H:", lam)
print(f"numerical rank = {np.linalg.matrix_rank(H, tol=1e-12 * lam.max())}   (n - 1 = {n - 1})")
assert np.linalg.matrix_rank(H, tol=1e-12 * lam.max()) == n - 1
assert lam.min() > -1e-14 and abs(lam[0]) < 1e-14

# Var_S(v) = 0 only for constant v: that, not orthogonality to 1, pins the nullspace.
v_const = 2.4 * np.ones(n)
v_orth = rng.normal(size=n); v_orth -= v_orth.mean()          # orthogonal to 1
print(f"v = c*1        : v^T H v = {float(v_const @ H @ v_const):.3e}")
print(f"v perp to 1    : v^T H v = {float(v_orth @ H @ v_orth):.6f}  (strictly positive)")
assert float(v_orth @ H @ v_orth) > 1e-6

L2.2  gradient vs S - e_y                            residual = 3.512e-10
L2.2  Hessian vs diag(S) - S S^T                     residual = 2.444e-08

eigenvalues of H: [-0.      0.055   0.0823  0.114   0.2056  0.3054]
numerical rank = 5   (n - 1 = 5)
v = c*1        : v^T H v = -1.332e-15
v perp to 1    : v^T H v = 1.163898  (strictly positive)


---

### Problem L2.3 — Gradient Descent Convergence Rate on Quadratic Objectives
**Source:** Polyak, *Introduction to Optimization* (1987, Ch. 1).  
**Problem Statement:**  
Consider $f(x) = \frac{1}{2} x^T H x$ for symmetric $H \succ 0$ with eigenvalues $0 \lt m = \lambda_{\min} \le \dots \le \lambda_{\max} = L$. For gradient descent $x^{(k+1)} = x^{(k)} - \alpha \nabla f(x^{(k)})$, find the optimal step size $\alpha^\ast$ and the minimal error contraction factor $\rho^\ast$.

#### First-Principles Intuition
Gradient descent iteration takes the operator form $x^{(k+1)} = (I - \alpha H) x^{(k)}$. The optimal step size minimizes the maximum absolute eigenvalue of the iteration operator matrix $(I - \alpha H)$.

#### Step-by-Step Solution
1. **Iteration Operator Matrix:**

$$
   \nabla f(x) = H x \implies x^{(k+1)} = x^{(k)} - \alpha H x^{(k)} = (I - \alpha H) x^{(k)}
$$

2. **Eigenvalues of Iteration Matrix:**
   The eigenvalues of $I - \alpha H$ are $\mu_i = 1 - \alpha \lambda_i$.  
   To guarantee convergence, we require $\lvert\mu_i\rvert \lt 1$ for all $i$, meaning:

$$
   -1 \lt 1 - \alpha \lambda_{\max} \le 1 - \alpha \lambda_{\min} \lt 1 \implies \alpha \lt \frac{2}{\lambda_{\max}} = \frac{2}{L}
$$

3. **Optimize Step Size $\alpha^\ast$:**
   The spectral radius $\rho(\alpha) = \max_i \lvert 1 - \alpha \lambda_i \rvert = \max(\lvert 1 - \alpha m \rvert, \lvert 1 - \alpha L \rvert)$.  
   The minimum worst-case contraction occurs when the two boundary error bounds balance:

$$
   1 - \alpha^\ast m = -(1 - \alpha^\ast L) \implies 2 = \alpha^\ast (L + m) \implies \alpha^\ast = \frac{2}{L + m}
$$

4. **Compute Minimal Contraction Factor $\rho^\ast$:**

$$
   \rho^\ast = 1 - \alpha^\ast m = 1 - \frac{2m}{L + m} = \frac{L - m}{L + m}
$$

   Divide numerator and denominator by $m$, noting condition number $\kappa = \frac{L}{m}$:

$$
\boxed{\alpha^\ast = \frac{2}{\lambda_{\max} + \lambda_{\min}}, \quad \rho^\ast = \frac{\kappa - 1}{\kappa + 1}}
$$

#### Key Insight / Takeaway
When $\kappa \gg 1$, $\rho^\ast \approx 1 - \frac{2}{\kappa}$, proving that ill-conditioned Hessians severely retard first-order gradient descent convergence rates.

**Verification.** The cell below recomputes the boxed answer of Problem L2.3 and asserts agreement.

In [22]:
# Optimal step and contraction factor for gradient descent on a quadratic.
n = 6
Q = np.linalg.qr(rng.normal(size=(n, n)))[0]
lam_true = np.array([0.4, 0.9, 1.6, 3.0, 5.5, 8.0])
H = Q @ np.diag(lam_true) @ Q.T
H = 0.5 * (H + H.T)
lam = np.linalg.eigvalsh(H)
m_, L_ = lam.min(), lam.max()
kappa = L_ / m_
alpha_star = 2.0 / (L_ + m_)
rho_star = (kappa - 1) / (kappa + 1)
print(f"m = {m_:.4f}  L = {L_:.4f}  kappa = {kappa:.4f}")
print(f"alpha* = {alpha_star:.8f}   rho* = {rho_star:.8f}")

# alpha* really minimises the spectral radius of I - alpha H.
spec = lambda a: np.abs(1 - a * lam).max()
grid = np.linspace(1e-4, 2.0 / L_ - 1e-6, 200001)
a_best = grid[np.argmin([spec(a) for a in grid])]
report("L2.3  argmin of the spectral radius", a_best, alpha_star, tol=1e-4)
report("L2.3  min spectral radius", spec(alpha_star), rho_star, tol=1e-12)

# Worst case: the error lives in the two extreme eigendirections, where the contraction is
# exactly rho* at every single step (one mode multiplies by +rho*, the other by -rho*).
U = np.linalg.eigh(H)[1]
x = U[:, 0] + U[:, -1]
ratios = []
for _ in range(40):
    x_next = x - alpha_star * (H @ x)
    ratios.append(np.linalg.norm(x_next) / np.linalg.norm(x))
    x = x_next
ratios = np.array(ratios)
print(f"\nper-step contraction on the worst-case start: min {ratios.min():.8f}, max {ratios.max():.8f}")
report("L2.3  per-step contraction vs rho*", ratios, np.full(40, rho_star), tol=1e-10)

# A generic start contracts at the same asymptotic rate once the fast modes have died.
x = rng.normal(size=n)
tail = []
for k in range(400):
    x_next = x - alpha_star * (H @ x)
    if k >= 350:
        tail.append(np.linalg.norm(x_next) / np.linalg.norm(x))
    x = x_next
print(f"generic start, per-step contraction over steps 350-399: {np.mean(tail):.8f}")
assert abs(np.mean(tail) - rho_star) < 1e-3

m = 0.4000  L = 8.0000  kappa = 20.0000
alpha* = 0.23809524   rho* = 0.90476190


L2.3  argmin of the spectral radius                  residual = 1.770e-07
L2.3  min spectral radius                            residual = 0.000e+00

per-step contraction on the worst-case start: min 0.90476190, max 0.90476190
L2.3  per-step contraction vs rho*                   residual = 3.331e-16
generic start, per-step contraction over steps 350-399: 0.90476190


---

### Problem L2.4 — Single-Step Convergence of Newton's Method on Quadratic Functions
**Source:** Nocedal & Wright, *Numerical Optimization* (Ch. 3).  
**Problem Statement:**  
Let $f(x) = \frac{1}{2} x^T A x - b^T x + c$ with symmetric $A \succ 0$. Show that starting from any initial point $x_0 \in \mathbb{R}^n$, one step of multivariable Newton's method yields the exact minimizer $x^\ast = A^{-1} b$.

#### First-Principles Intuition
Newton's method uses a local 2nd-order Taylor model. For a quadratic function, the 2nd-order Taylor model is globally exact everywhere.

#### Step-by-Step Solution
1. **Gradient and Hessian of $f(x)$:**

$$
   \nabla f(x) = A x - b
$$

$$
   H_f(x) = A \quad (\text{constant for all } x)
$$

2. **Apply Newton Update from $x_0$:**

$$
   x_1 = x_0 - H_f(x_0)^{-1} \nabla f(x_0) = x_0 - A^{-1} (A x_0 - b)
$$

3. **Expand Matrix Multiplication:**

$$
   x_1 = x_0 - (A^{-1} A x_0 - A^{-1} b) = x_0 - (x_0 - A^{-1} b) = A^{-1} b
$$

4. **Verify Global Minimum:**
   Since $A \succ 0$, $x^\ast = A^{-1} b$ satisfies $\nabla f(x^\ast) = A(A^{-1} b) - b = \mathbf{0}$.

$$
\boxed{x_1 = A^{-1} b = x^\ast \quad (\text{exact global minimizer in 1 step})}
$$

#### Key Insight / Takeaway
Newton's method achieves exact single-step convergence on quadratic functions because the Hessian contains complete higher-order curvature information.

**Verification.** The cell below recomputes the boxed answer of Problem L2.4 and asserts agreement.

In [23]:
# Newton reaches A^{-1}b in exactly one step, from any start.
n = 7
B = rng.normal(size=(n, n))
A = B @ B.T + n * np.eye(n)
b = rng.normal(size=n)
x_star = np.linalg.solve(A, b)
grad = lambda v: A @ v - b
for x0 in (np.zeros(n), rng.normal(size=n), 1e3 * rng.normal(size=n)):
    x1 = x0 - np.linalg.solve(A, grad(x0))
    report(f"L2.4  one Newton step from ||x0|| = {np.linalg.norm(x0):9.3f}", x1, x_star, tol=1e-9)
print("\nThe distance to the minimiser is killed in one step because the quadratic model is exact.")
report("L2.4  gradient at x*", grad(x_star), np.zeros(n), tol=1e-10)

L2.4  one Newton step from ||x0|| =     0.000        residual = 0.000e+00
L2.4  one Newton step from ||x0|| =     2.025        residual = 1.874e-16
L2.4  one Newton step from ||x0|| =  3143.308        residual = 5.555e-13

The distance to the minimiser is killed in one step because the quadratic model is exact.
L2.4  gradient at x*                                 residual = 1.110e-16


---

### Problem L2.5 — Hessian and Convexity of Logistic Regression Loss
**Source:** Boyd & Vandenberghe, *Convex Optimization* (Ch. 3).  
**Problem Statement:**  
The binary logistic regression loss for dataset $\{(x_i, y_i)\}_{i=1}^N$ with $y_i \in \{-1, +1\}$ is:

$$
\mathcal{L}(w) = \sum_{i=1}^N \ln\left(1 + e^{-y_i w^T x_i}\right)
$$

Derive the gradient $\nabla \mathcal{L}(w)$ and Hessian $H(w)$, and prove that $\mathcal{L}(w)$ is convex.

#### First-Principles Intuition
The loss is a sum of softplus functions evaluated at linear predictions. Since softplus is convex and linear compositions preserve convexity, the loss Hessian must be positive semi-definite.

#### Step-by-Step Solution
Let $z_i = y_i w^T x_i$, and define sigmoid $\sigma(z) = \frac{1}{1 + e^{-z}}$.
Note that $\frac{d}{dz} \ln(1 + e^{-z}) = \frac{-e^{-z}}{1 + e^{-z}} = -\sigma(-z) = \sigma(z) - 1$.

1. **Compute Gradient $\nabla \mathcal{L}(w)$:**

$$
   \nabla \mathcal{L}(w) = \sum_{i=1}^N \frac{-y_i x_i e^{-y_i w^T x_i}}{1 + e^{-y_i w^T x_i}} = -\sum_{i=1}^N y_i x_i \sigma(-y_i w^T x_i)
$$

2. **Compute Hessian Matrix $H(w)$:**
   Differentiate $\nabla \mathcal{L}(w)$ with respect to $w^T$:
   Recall $\sigma'(z) = \sigma(z)(1 - \sigma(z)) = p_i (1 - p_i)$, where $p_i = \sigma(y_i w^T x_i)$.

$$
   H(w) = \sum_{i=1}^N y_i^2 x_i x_i^T \sigma(y_i w^T x_i)(1 - \sigma(y_i w^T x_i)) = \sum_{i=1}^N p_i (1 - p_i) x_i x_i^T
$$

3. **Prove Positive Semi-Definiteness:**
   For any vector $v \in \mathbb{R}^d$:

$$
   v^T H(w) v = v^T \left( \sum_{i=1}^N p_i (1 - p_i) x_i x_i^T \right) v = \sum_{i=1}^N p_i (1 - p_i) (v^T x_i)^2
$$

   Since $p_i \in (0, 1)$, $p_i (1 - p_i) \gt 0$. Also $(v^T x_i)^2 \ge 0$.  
   Therefore, $v^T H(w) v \ge 0$ for all $v$, proving $H(w) \succeq 0$, so $\mathcal{L}$ is convex.

4. **When is it strict?**
   Every weight $p_i(1-p_i)$ is strictly positive, so $v^\top H(w) v = 0$ holds **iff** $v^\top x_i = 0$ for every $i$, i.e. iff $v \in \operatorname{Null}(X)$ where $X \in \mathbb{R}^{N \times d}$ stacks the $x_i^\top$. Hence

$$
   H(w) \succ 0 \iff \operatorname{Null}(X) = \lbrace \mathbf{0} \rbrace \iff \operatorname{rank}(X) = d .
$$

   With $N \lt d$ — the over-parameterised regime — $\operatorname{rank}(X) \le N \lt d$, so $H(w)$ is singular and $\mathcal{L}$ is convex but **not** strictly convex: the loss is exactly flat along $\operatorname{Null}(X)$ and the minimiser is not unique.

$$
\boxed{H(w) = X^\top D X \succeq 0, \quad D = \operatorname{diag}\big(p_i(1-p_i)\big) \succ 0; \qquad H(w) \succ 0 \iff \operatorname{rank}(X) = d}
$$

#### Key Insight / Takeaway
Logistic regression loss has a positive semi-definite Hessian equal to a weighted data covariance $X^\top D X$, so there are no non-global local minima; strict convexity is an extra assumption on the design matrix, not a free consequence of the model.

**Verification.** The cell below recomputes the boxed answer of Problem L2.5 and asserts agreement.

In [24]:
# Logistic loss: Hessian X^T D X, PSD always, PD exactly when X has full column rank.
sigma = lambda t: 1.0 / (1.0 + np.exp(-t))

def logistic_pieces(X, yv, w):
    z = yv * (X @ w)
    p = sigma(z)
    loss = np.log1p(np.exp(-z)).sum()
    grad = -(X * (yv * (1 - p))[:, None]).sum(axis=0)
    Hs = X.T @ np.diag(p * (1 - p)) @ X
    return loss, grad, Hs

N, d = 40, 5
X = rng.normal(size=(N, d))
yv = rng.choice([-1.0, 1.0], size=N)
w = rng.normal(size=d) * 0.5
loss, grad, H = logistic_pieces(X, yv, w)

h = 1e-6
g_fd = np.array([(logistic_pieces(X, yv, w + h * np.eye(d)[j])[0]
                  - logistic_pieces(X, yv, w - h * np.eye(d)[j])[0]) / (2 * h) for j in range(d)])
report("L2.5  gradient vs finite differences", g_fd, grad, tol=1e-6)
H_fd = np.array([(logistic_pieces(X, yv, w + h * np.eye(d)[j])[1]
                  - logistic_pieces(X, yv, w - h * np.eye(d)[j])[1]) / (2 * h) for j in range(d)]).T
report("L2.5  Hessian vs finite differences", H_fd, H, tol=1e-5)
print(f"\nfull column rank (N = {N} > d = {d}): lambda_min(H) = {np.linalg.eigvalsh(H).min():.6e}  -> H > 0")
assert np.linalg.eigvalsh(H).min() > 0

# Rank-deficient design: still PSD, no longer PD, so the loss is convex but not strictly convex.
X2 = rng.normal(size=(3, d))                    # N = 3 < d = 5
y2 = rng.choice([-1.0, 1.0], size=3)
H2 = logistic_pieces(X2, y2, w)[2]
lam2 = np.linalg.eigvalsh(H2)
print(f"N = 3 < d = 5: eigenvalues = {lam2}")
print(f"lambda_min = {lam2.min():.3e}  -> PSD but singular: the loss is NOT strictly convex here.")
assert lam2.min() > -1e-12 and abs(lam2.min()) < 1e-10

L2.5  gradient vs finite differences                 residual = 1.587e-09
L2.5  Hessian vs finite differences                  residual = 2.733e-09

full column rank (N = 40 > d = 5): lambda_min(H) = 3.232300e+00  -> H > 0
N = 3 < d = 5: eigenvalues = [-0.     -0.      0.0452  0.3041  1.991 ]
lambda_min = -1.840e-16  -> PSD but singular: the loss is NOT strictly convex here.


---

### Problem L2.6 — Jacobian & Local Volume Distortion of Planar Elastic Deformation
**Source:** Adapted from Demidovich No. 3160 & Spivak.  
**Problem Statement:**  
An elastic planar deformation mapping $(x, y) \to (u, v)$ is given by $u(x,y) = x + \epsilon \sin y$ and $v(x,y) = y + \epsilon \cos x$, where $\epsilon \gt 0$ is a small strain parameter. Compute the Jacobian matrix $J$, its determinant $\det(J)$, and find the local area expansion factor to first order in $\epsilon$.

#### First-Principles Intuition
The Jacobian determinant measures local area dilatation (volume change) during non-linear physical deformation.

#### Step-by-Step Solution
1. **Compute Jacobian Matrix $J$:**

$$
   J = \begin{bmatrix} \frac{\partial u}{\partial x} & \frac{\partial u}{\partial y} \\[4pt] \frac{\partial v}{\partial x} & \frac{\partial v}{\partial y} \end{bmatrix} = \begin{bmatrix} 1 & \epsilon \cos y \\ -\epsilon \sin x & 1 \end{bmatrix}
$$

2. **Compute Jacobian Determinant $\det(J)$:**

$$
   \det(J) = (1)(1) - (\epsilon \cos y)(-\epsilon \sin x) = 1 + \epsilon^2 \sin x \cos y
$$

3. **First-Order Expansion in $\epsilon$:**
   To first order in $\epsilon$ ($\mathcal{O}(\epsilon^2) \to 0$):

$$
   \det(J) \approx 1 + 0 \cdot \epsilon = 1
$$

$$
\boxed{\det(J) = 1 + \epsilon^2 \sin x \cos y \approx 1 + \mathcal{O}(\epsilon^2) \quad (\text{infinitesimal area is preserved to 1st order}) }
$$

#### Key Insight / Takeaway
Small non-linear shear deformations preserve volume to first order in strain $\epsilon$, with area distortion appearing only at second order $\epsilon^2$.

**Verification.** The cell below recomputes the boxed answer of Problem L2.6 and asserts agreement.

In [25]:
# Elastic deformation: det J = 1 + eps^2 sin x cos y, so the O(eps) term is absent.
x, y, eps = sp.symbols("x y epsilon", real=True)
Jd = sp.Matrix([x + eps * sp.sin(y), y + eps * sp.cos(x)]).jacobian([x, y])
det_d = sp.simplify(Jd.det())
print("J =", Jd.tolist())
print("det J =", det_d)
assert sp.simplify(det_d - (1 + eps**2 * sp.sin(x) * sp.cos(y))) == 0
print("series in eps:", sp.series(det_d, eps, 0, 3))
assert sp.simplify(sp.diff(det_d, eps).subs(eps, 0)) == 0

# Measured area gain of a small square, versus the eps^2 prediction.
x0, y0, s = 0.9, 0.4, 1e-4
def shoelace(p):
    a, b = p[:, 0], p[:, 1]
    return 0.5 * abs(np.dot(a, np.roll(b, -1)) - np.dot(b, np.roll(a, -1)))
for e in (1e-1, 1e-2, 1e-3):
    F = lambda u, v: np.array([u + e * np.sin(v), v + e * np.cos(u)])
    sq = np.array([F(x0, y0), F(x0 + s, y0), F(x0 + s, y0 + s), F(x0, y0 + s)])
    gain = shoelace(sq) / s**2
    pred = 1 + e**2 * np.sin(x0) * np.cos(y0)
    print(f"  eps = {e:.0e}:  measured area gain = {gain:.10f}   predicted = {pred:.10f}")
    assert abs(gain - pred) < 1e-6

J = [[1, epsilon*cos(y)], [-epsilon*sin(x), 1]]
det J = epsilon**2*sin(x)*cos(y) + 1
series in eps: epsilon**2*sin(x)*cos(y) + 1
  eps = 1e-01:  measured area gain = 1.0072150558   predicted = 1.0072149186
  eps = 1e-02:  measured area gain = 1.0000721584   predicted = 1.0000721492
  eps = 1e-03:  measured area gain = 1.0000007156   predicted = 1.0000007215


---

### Problem L2.7 — Natural Gradient Descent & KL-Divergence Second-Order Expansion
**Source:** Amari, *Information Geometry and Its Applications*.  
**Problem Statement:**  
Show that the second-order Taylor expansion of the Kullback-Leibler divergence $\mathbb{D}_{KL}(p_\theta \parallel p_{\theta + d\theta})$ around $d\theta = \mathbf{0}$ is $\frac{1}{2} d\theta^T F(\theta) d\theta$, where $F(\theta) = \mathbb{E}_{x \sim p_\theta}[\nabla_\theta \ln p_\theta(x) \nabla_\theta \ln p_\theta(x)^T]$ is the Fisher Information Matrix.

#### First-Principles Intuition
The KL divergence measures statistical distance between probability distributions. Since $\mathbb{D}_{KL}(p_\theta \parallel p_\theta) = 0$ is a global minimum, its first derivative vanishes, making its Hessian (the Fisher Information Matrix) the primary measure of distance.

#### Step-by-Step Solution
Define $g(d\theta) = \mathbb{D}_{KL}(p_\theta \parallel p_{\theta + d\theta}) = \int p_\theta(x) \ln \frac{p_\theta(x)}{p_{\theta + d\theta}(x)} \, dx$.

1. **Evaluate at $d\theta = \mathbf{0}$:**

$$
   g(\mathbf{0}) = \int p_\theta(x) \ln 1 \, dx = 0
$$

2. **First Derivative at $d\theta = \mathbf{0}$:**

$$
   \nabla_{d\theta} g(\mathbf{0}) = -\int p_\theta(x) \nabla_\theta \ln p_\theta(x) \, dx = -\int \nabla_\theta p_\theta(x) \, dx = -\nabla_\theta \left( \int p_\theta(x) \, dx \right) = -\nabla_\theta(1) = \mathbf{0}
$$

3. **Second Derivative (Hessian Matrix) at $d\theta = \mathbf{0}$:**

$$
   \nabla_{d\theta}^2 g(d\theta) = -\int p_\theta(x) \nabla_{\theta + d\theta}^2 \ln p_{\theta + d\theta}(x) \, dx
$$

   At $d\theta = \mathbf{0}$:

$$
   \nabla_{d\theta}^2 g(\mathbf{0}) = -\mathbb{E}_{x \sim p_\theta} \left[ \nabla_\theta^2 \ln p_\theta(x) \right]
$$

   Using the identity $\nabla^2 \ln p = \frac{\nabla^2 p}{p} - \frac{\nabla p \nabla p^T}{p^2} = \frac{\nabla^2 p}{p} - \nabla \ln p \nabla \ln p^T$:

$$
   -\mathbb{E}\left[\nabla^2 \ln p\right] = -\int \nabla^2 p(x) \, dx + \mathbb{E}\left[\nabla \ln p \nabla \ln p^T\right] = \mathbf{0} + F(\theta) = F(\theta)
$$

$$
\boxed{\mathbb{D}_{KL}(p_\theta \parallel p_{\theta + d\theta}) = \frac{1}{2} d\theta^T F(\theta) d\theta + \mathcal{O}(\lVert d\theta\rVert^3)}
$$

#### Key Insight / Takeaway
The Fisher Information Matrix is the Hessian of KL-divergence, establishing Riemannian curvature on statistical probability manifolds.

**Verification.** The cell below recomputes the boxed answer of Problem L2.7 and asserts agreement.

In [26]:
# KL(p_theta || p_{theta+d}) = 0.5 d^T F d + O(||d||^3), on the Gaussian family N(mu, sigma^2).
# Exact KL for two 1-D Gaussians, and the exact Fisher matrix F = diag(1/s^2, 2/s^2).
def kl_gauss(m0, s0, m1, s1):
    return np.log(s1 / s0) + (s0**2 + (m0 - m1) ** 2) / (2 * s1**2) - 0.5

mu0, sig0 = 0.7, 1.3
F = np.array([[1 / sig0**2, 0.0], [0.0, 2 / sig0**2]])
print("Fisher matrix F =\n", F)

u = rng.normal(size=2)
u /= np.linalg.norm(u)
print("\n  t          KL(exact)        0.5 t^2 u^T F u      ratio")
prev = None
for t in (1e-1, 1e-2, 1e-3, 1e-4):
    d = t * u
    kl = kl_gauss(mu0, sig0, mu0 + d[0], sig0 + d[1])
    quad = 0.5 * d @ F @ d
    print(f"  {t:.0e}   {kl:.12e}   {quad:.12e}   {kl/quad:.9f}")
    if prev is not None:
        assert abs(kl / quad - 1) < abs(prev) / 5      # error shrinks like t, i.e. O(||d||^3)
    prev = kl / quad - 1
assert abs(kl / quad - 1) < 5e-4
print("\nThe ratio approaches 1 linearly in t, which is exactly the O(||d||^3) remainder.")

# F is also the expected outer product of the score, and the Hessian of the KL at d = 0.
xs = rng.normal(loc=mu0, scale=sig0, size=2_000_000)
score = np.stack([(xs - mu0) / sig0**2, ((xs - mu0) ** 2 - sig0**2) / sig0**3])
F_mc = score @ score.T / xs.size
print("\nMonte-Carlo E[score score^T] =\n", F_mc)
assert np.abs(F_mc - F).max() < 5e-3

Fisher matrix F =
 [[0.5917 0.    ]
 [0.     1.1834]]

  t          KL(exact)        0.5 t^2 u^T F u      ratio
  1e-01   6.312548705761e-03   5.561977863839e-03   1.134946751
  1e-02   5.630394030109e-05   5.561977863839e-05   1.012300690
  1e-03   5.568758988383e-07   5.561977863839e-07   1.001219193
  1e-04   5.562655447378e-09   5.561977863839e-09   1.000121824

The ratio approaches 1 linearly in t, which is exactly the O(||d||^3) remainder.



Monte-Carlo E[score score^T] =
 [[ 0.5913 -0.0015]
 [-0.0015  1.1835]]


---

### Problem L2.8 — Hessian & Eigenvalue Lower Bound for Ridge Regression Loss
**Source:** Adapted from Nocedal & Wright (Ch. 2) & Boyd.  
**Problem Statement:**  
Consider the Ridge Regression loss function $\mathcal{L}(w) = \frac{1}{2} \lVert X w - y\rVert^2 + \frac{\lambda}{2} \lVert w\rVert^2$, where $X \in \mathbb{R}^{N \times p}$ and $\lambda \gt 0$. Compute the Hessian $H(w)$, show that $H(w) \succ 0$ even if $X^T X$ is singular ($N \lt p$), and find the lower bound on its smallest eigenvalue.

#### First-Principles Intuition
Adding L2 regularization $\frac{\lambda}{2} \lVert w\rVert^2$ shifts all eigenvalues of the Gram matrix $X^T X$ upward by $\lambda$, guaranteeing strict positive definiteness and nonsingularity.

#### Step-by-Step Solution
1. **Compute Gradient & Hessian:**
   - Loss gradient: $\nabla \mathcal{L}(w) = X^T (X w - y) + \lambda w = (X^T X + \lambda I) w - X^T y$.
   - Hessian matrix: $H(w) = \nabla^2 \mathcal{L}(w) = X^T X + \lambda I$.

2. **Show $H(w) \succ 0$:**
   For any non-zero vector $v \in \mathbb{R}^p$:

$$
   v^T H(w) v = v^T (X^T X + \lambda I) v = (X v)^T (X v) + \lambda v^T v = \lVert X v\rVert^2 + \lambda \lVert v\rVert^2
$$

   Since $\lVert X v\rVert^2 \ge 0$ and $\lambda \lVert v\rVert^2 \gt 0$ for $v \neq \mathbf{0}$, $v^T H(w) v \gt 0$.

3. **Eigenvalue Lower Bound:**
   Let $\sigma_1 \ge \dots \ge \sigma_p \ge 0$ be singular values of $X$. Eigenvalues of $X^T X$ are $\lambda_i(X^T X) = \sigma_i^2 \ge 0$.  
   Eigenvalues of $H(w)$ are $\mu_i = \sigma_i^2 + \lambda$.  
   Since $\sigma_i^2 \ge 0$ and $\lambda \gt 0$:

$$
   \mu_{\min} = \sigma_{\min}^2 + \lambda \ge \lambda \gt 0
$$

   Thus $v^T H v \ge \lambda \lVert v\rVert^2 \gt 0$ for all non-zero $v$, guaranteeing $H(w) \succ 0$.

$$
\boxed{H(w) = X^T X + \lambda I, \quad \lambda_{\min}(H) \ge \lambda \gt 0 \implies H(w) \text{ is strictly positive definite}}
$$

#### Key Insight / Takeaway
Tikhonov / Ridge regularization adds isotropic positive curvature $\lambda I$ to the Hessian, rendering ill-conditioned or rank-deficient quadratic objectives strictly convex.

**Verification.** The cell below recomputes the boxed answer of Problem L2.8 and asserts agreement.

In [27]:
# Ridge Hessian X^T X + lambda I: positive definite even when X^T X is singular.
N, p, lam_reg = 6, 20, 0.35          # N < p, so X^T X is rank deficient
X = rng.normal(size=(N, p))
H = X.T @ X + lam_reg * np.eye(p)
ev_gram = np.linalg.eigvalsh(X.T @ X)
ev_H = np.linalg.eigvalsh(H)
print(f"rank(X^T X) = {np.linalg.matrix_rank(X.T @ X)}  (p = {p}), lambda_min(X^T X) = {ev_gram.min():.3e}")
print(f"lambda_min(H) = {ev_H.min():.6f}   lambda (the regulariser) = {lam_reg}")
report("L2.8  spectrum of H vs spectrum of X^T X shifted by lambda", ev_H, ev_gram + lam_reg, tol=1e-9)
assert ev_H.min() >= lam_reg - 1e-9
sv = np.linalg.svd(X, compute_uv=False)
report("L2.8  largest eigenvalue vs sigma_max^2 + lambda", ev_H.max(), sv.max() ** 2 + lam_reg, tol=1e-9)
v = rng.normal(size=p)
print(f"\nv^T H v = {float(v @ H @ v):.6f}  >=  lambda ||v||^2 = {lam_reg * v @ v:.6f}")
assert float(v @ H @ v) >= lam_reg * float(v @ v) - 1e-9

rank(X^T X) = 6  (p = 20), lambda_min(X^T X) = -5.792e-15
lambda_min(H) = 0.350000   lambda (the regulariser) = 0.35
L2.8  spectrum of H vs spectrum of X^T X shifted by lambda residual = 1.776e-14
L2.8  largest eigenvalue vs sigma_max^2 + lambda     residual = 2.842e-14

v^T H v = 281.986558  >=  lambda ||v||^2 = 9.047355


---

### Problem L2.9 — Fisher Information of a Gaussian as an Expected Hessian
**Source:** Adapted from Amari, *Information Geometry and Its Applications* & Bishop, *PRML* (Ch. 1).  
**Problem Statement:**  
Let $p_\theta(x) = \mathcal{N}(x \mid \mu, \sigma^2)$ on $\mathbb{R}$ with parameters $\theta = (\mu, \sigma)^\top$, $\sigma \gt 0$, and let $\ell(\theta; x) = -\ln p_\theta(x)$ be the per-sample negative log-likelihood. Compute the Hessian $\nabla_\theta^2 \ell(\theta; x)$, take its expectation under $x \sim p_\theta$ to obtain the Fisher information matrix $F(\theta)$, and write down the natural-gradient step $\Delta\theta = -\eta \, F(\theta)^{-1} \nabla_\theta \ell$.

#### First-Principles Intuition
The Hessian of the loss at a single sample still depends on that sample. Averaging over $x \sim p_\theta$ annihilates every term that is odd in $x - \mu$ and leaves a matrix depending on $\theta$ alone: the curvature the model *expects*, which by Problem L2.7 is also the local metric of KL divergence.

#### Step-by-Step Solution
1. **Write the negative log-likelihood:**

$$
   \ell(\theta; x) = \ln \sigma + \frac{(x - \mu)^2}{2\sigma^2} + \tfrac{1}{2}\ln(2\pi)
$$

2. **First derivatives (the score, up to sign):**

$$
   \frac{\partial \ell}{\partial \mu} = -\frac{x - \mu}{\sigma^2},
   \qquad
   \frac{\partial \ell}{\partial \sigma} = \frac{1}{\sigma} - \frac{(x - \mu)^2}{\sigma^3}
$$

3. **Second derivatives:**

$$
   \frac{\partial^2 \ell}{\partial \mu^2} = \frac{1}{\sigma^2},
   \qquad
   \frac{\partial^2 \ell}{\partial \mu \, \partial \sigma} = \frac{2(x - \mu)}{\sigma^3},
   \qquad
   \frac{\partial^2 \ell}{\partial \sigma^2} = -\frac{1}{\sigma^2} + \frac{3(x - \mu)^2}{\sigma^4}
$$

   so the sample Hessian is

$$
   \nabla_\theta^2 \ell(\theta; x)
   = \begin{bmatrix}
   \dfrac{1}{\sigma^2} & \dfrac{2(x-\mu)}{\sigma^3} \\[8pt]
   \dfrac{2(x-\mu)}{\sigma^3} & -\dfrac{1}{\sigma^2} + \dfrac{3(x-\mu)^2}{\sigma^4}
   \end{bmatrix}.
$$

   It is symmetric because $\ell$ is $C^\infty$ in $\theta$ on $\sigma \gt 0$, so Theorem 4.1 applies.

4. **Take the expectation under $x \sim \mathcal{N}(\mu, \sigma^2)$:**
   Using $\mathbb{E}[x - \mu] = 0$ and $\mathbb{E}[(x-\mu)^2] = \sigma^2$, the off-diagonal entry averages to $0$ and

$$
   \mathbb{E}\left[ -\frac{1}{\sigma^2} + \frac{3(x-\mu)^2}{\sigma^4} \right]
   = -\frac{1}{\sigma^2} + \frac{3\sigma^2}{\sigma^4} = \frac{2}{\sigma^2}.
$$

5. **Natural gradient:**
   $F$ is diagonal, so its inverse is $\operatorname{diag}(\sigma^2, \sigma^2/2)$ and the natural-gradient step rescales each coordinate by the variance it lives on:

$$
   \Delta\theta = -\eta F^{-1} \nabla_\theta \ell
   = -\eta \left( \sigma^2 \frac{\partial \ell}{\partial \mu}, \ \frac{\sigma^2}{2}\frac{\partial \ell}{\partial \sigma} \right)^{\!\top}.
$$

$$
\boxed{F(\theta) = \mathbb{E}_{x \sim p_\theta}\!\left[\nabla_\theta^2 \ell\right] = \begin{bmatrix} \dfrac{1}{\sigma^2} & 0 \\[6pt] 0 & \dfrac{2}{\sigma^2} \end{bmatrix}, \qquad F^{-1}\nabla_\theta \ell = \left(\sigma^2 \partial_\mu \ell, \ \tfrac{\sigma^2}{2}\partial_\sigma \ell\right)^{\!\top}}
$$

#### Key Insight / Takeaway
The Fisher matrix is the *expected* Hessian of the negative log-likelihood; it is what makes a gradient step scale-aware, since a step of $0.1$ in $\mu$ means something very different when $\sigma = 0.01$ than when $\sigma = 100$.

**Verification.** The cell below recomputes the boxed answer of Problem L2.9 and asserts agreement.

In [28]:
# Fisher matrix of N(mu, sigma) two ways, and the natural-gradient rescaling.
mu, sg, xv = sp.symbols("mu sigma x", real=True), None, None
mu = sp.Symbol("mu", real=True); sg = sp.Symbol("sigma", positive=True); xv = sp.Symbol("x", real=True)
nll = sp.log(sg) + (xv - mu) ** 2 / (2 * sg**2) + sp.log(sp.sqrt(2 * sp.pi))
Hs = sp.simplify(sp.hessian(nll, (mu, sg)))
print("Hessian of the per-sample NLL:\n", Hs.tolist())
dens = sp.exp(-(xv - mu) ** 2 / (2 * sg**2)) / (sg * sp.sqrt(2 * sp.pi))
F_sym = sp.simplify(sp.Matrix(2, 2, lambda i, j: sp.integrate(Hs[i, j] * dens, (xv, -sp.oo, sp.oo))))
print("F = E[Hessian] =", F_sym.tolist())
assert sp.simplify(F_sym - sp.diag(1 / sg**2, 2 / sg**2)) == sp.zeros(2, 2)

# Monte-Carlo score covariance must match the same matrix.
mu0, sig0 = -0.4, 0.8
F_num = np.diag([1 / sig0**2, 2 / sig0**2])
xs = rng.normal(loc=mu0, scale=sig0, size=4_000_000)
score = np.stack([(xs - mu0) / sig0**2, ((xs - mu0) ** 2 - sig0**2) / sig0**3])
F_mc = score @ score.T / xs.size
print("\nMonte-Carlo E[score score^T] =\n", F_mc)
print("closed form F                =\n", F_num)
assert np.abs(F_mc - F_num).max() < 5e-3

# Natural gradient: F^{-1} grad is invariant to the sigma-scale that distorts the plain gradient.
g = np.array([0.5, 0.5])
print(f"\nEuclidean gradient : {g}")
print(f"natural gradient   : {np.linalg.solve(F_num, g)}  = sigma^2 * (g_mu, g_sigma/2)")
report("L2.9  F^{-1} g", np.linalg.solve(F_num, g), np.array([sig0**2 * 0.5, sig0**2 * 0.25]), tol=1e-12)

Hessian of the per-sample NLL:
 [[sigma**(-2), 2*(-mu + x)/sigma**3], [2*(-mu + x)/sigma**3, (-sigma**2 + 3*(mu - x)**2)/sigma**4]]


F = E[Hessian] = [[sigma**(-2), 0], [0, 2/sigma**2]]

Monte-Carlo E[score score^T] =
 [[1.5616 0.0007]
 [0.0007 3.1253]]
closed form F                =
 [[1.5625 0.    ]
 [0.     3.125 ]]

Euclidean gradient : [0.5 0.5]
natural gradient   : [0.32 0.16]  = sigma^2 * (g_mu, g_sigma/2)
L2.9  F^{-1} g                                       residual = 0.000e+00


---

### Problem L2.10 — Kinetic Energy Metric Tensor in Lagrangian Mechanics
**Source:** Arnold, *Mathematical Methods of Classical Mechanics*.  
**Problem Statement:**  
Consider a particle of mass $m$ moving in 2D polar coordinates $(r, \theta)$. The kinetic energy is $T(r, \theta, \dot{r}, \dot{\theta}) = \frac{1}{2} m (\dot{r}^2 + r^2 \dot{\theta}^2)$. Compute the Hessian of kinetic energy with respect to generalized velocities $g_{ij} = \frac{\partial^2 T}{\partial \dot{q}_i \partial \dot{q}_j}$ for $q = (r, \theta)^T$, and explain its physical significance.

#### First-Principles Intuition
In analytical mechanics, the second partial derivatives of kinetic energy with respect to generalized velocities define the mass/inertia matrix $M(q)$, which serves as the metric tensor of the configuration space.

#### Step-by-Step Solution
Let generalized velocities be $v_1 = \dot{r}$ and $v_2 = \dot{\theta}$.
Kinetic energy is $T(r, \theta, v_1, v_2) = \frac{1}{2} m v_1^2 + \frac{1}{2} m r^2 v_2^2$.

Compute second partial derivatives with respect to $(v_1, v_2)$:

$$
\frac{\partial^2 T}{\partial v_1^2} = \frac{\partial}{\partial v_1} (m v_1) = m
$$

$$
\frac{\partial^2 T}{\partial v_1 \partial v_2} = \frac{\partial}{\partial v_2} (m v_1) = 0
$$

$$
\frac{\partial^2 T}{\partial v_2^2} = \frac{\partial}{\partial v_2} (m r^2 v_2) = m r^2
$$

Construct the mass matrix / metric tensor $g$:

$$
\boxed{g = M(r, \theta) = \begin{bmatrix} m & 0 \\ 0 & m r^2 \end{bmatrix}}
$$

#### Key Insight / Takeaway
The generalized mass matrix in classical mechanics is the velocity-Hessian of kinetic energy, defining kinetic energy as a quadratic form $T = \frac{1}{2} \dot{q}^T M(q) \dot{q}$.

**Verification.** The cell below recomputes the boxed answer of Problem L2.10 and asserts agreement.

In [29]:
# Mass matrix as the velocity-Hessian of the kinetic energy, in polar coordinates.
m_s, r_s, v1, v2 = sp.symbols("m r v1 v2", positive=True)
T = sp.Rational(1, 2) * m_s * (v1**2 + r_s**2 * v2**2)
M = sp.simplify(sp.hessian(T, (v1, v2)))
print("M(q) = d^2T/dqdot^2 =", M.tolist())
assert sp.simplify(M - sp.diag(m_s, m_s * r_s**2)) == sp.zeros(2, 2)

# T = 0.5 qdot^T M qdot recovers the original expression exactly.
qd = sp.Matrix([v1, v2])
assert sp.simplify((sp.Rational(1, 2) * qd.T * M * qd)[0, 0] - T) == 0
print("0.5 qdot^T M qdot reproduces T exactly.")

# Numerical check, plus the link to the metric of Problem L1.10: M = m * g_polar.
mv, rv, vv = 2.3, 1.7, np.array([0.6, -0.9])
Mn = np.diag([mv, mv * rv**2])
T_direct = 0.5 * mv * (vv[0] ** 2 + rv**2 * vv[1] ** 2)
report("L2.10  0.5 v^T M v vs T", 0.5 * vv @ Mn @ vv, T_direct, tol=1e-12)
report("L2.10  M vs m * diag(1, r^2)", Mn, mv * np.diag([1.0, rv**2]), tol=1e-12)
print(f"\nangular inertia m r^2 = {mv * rv**2:.4f} grows with r: the same r^2 as the polar metric.")

M(q) = d^2T/dqdot^2 = [[m, 0], [0, m*r**2]]
0.5 qdot^T M qdot reproduces T exactly.
L2.10  0.5 v^T M v vs T                              residual = 0.000e+00
L2.10  M vs m * diag(1, r^2)                         residual = 0.000e+00

angular inertia m r^2 = 6.6470 grows with r: the same r^2 as the polar metric.


---

### Problem L2.11 — Hessian of Softplus Activation Function
**Source:** Goodfellow et al., *Deep Learning* (Ch. 6).  
**Problem Statement:**  
The Softplus function $\operatorname{Softplus}(x) = \ln(1 + e^x)$ is a smooth approximation of ReLU. Derive its derivative $f'(x)$ and second derivative $f''(x)$. For vector input $x \in \mathbb{R}^n$ with elementwise application $F(x)$, express its Hessian matrix $H_F(x)$.

#### First-Principles Intuition
Softplus smooths out the non-differentiable corner of $\max(0, x)$. Its first derivative is the Sigmoid function, and its second derivative is the Sigmoid probability density (variance factor).

#### Step-by-Step Solution
1. **Single-Variable Derivatives:**

$$
   f'(x) = \frac{d}{dx} \ln(1 + e^x) = \frac{e^x}{1 + e^x} = \sigma(x) \quad (\text{Sigmoid function})
$$

$$
   f''(x) = \frac{d}{dx} \sigma(x) = \sigma(x)(1 - \sigma(x))
$$

2. **Multivariable Vector Extension:**
   For $F(x) = (\operatorname{Softplus}(x_1), \dots, \operatorname{Softplus}(x_n))^T$, the elementwise activation mapping has a diagonal Jacobian matrix:

$$
   J_F(x) = \operatorname{diag}(\sigma(x_1), \sigma(x_2), \dots, \sigma(x_n))
$$

   If considering the scalar sum objective $S(x) = \sum_{i=1}^n \operatorname{Softplus}(x_i)$:

$$
   \nabla S(x) = \sigma(x) = \begin{bmatrix} \sigma(x_1) \\ \vdots \\ \sigma(x_n) \end{bmatrix}
$$

$$
   H_S(x) = \operatorname{diag}\Big( \sigma(x_1)(1 - \sigma(x_1)), \dots, \sigma(x_n)(1 - \sigma(x_n)) \Big)
$$

$$
   \boxed{f'(x) = \sigma(x), \quad f''(x) = \sigma(x)(1 - \sigma(x)), \quad H_S(x) = \operatorname{diag}(\sigma(x)(1 - \sigma(x))) \succ 0}
$$

#### Key Insight / Takeaway
Softplus is strictly convex everywhere because $f''(x) = \sigma(x)(1 - \sigma(x)) \gt 0$ for all real $x$, preventing zero-gradient dead zones.

**Verification.** The cell below recomputes the boxed answer of Problem L2.11 and asserts agreement.

In [30]:
# Softplus: f' = sigmoid, f'' = sigmoid(1 - sigmoid) > 0, and the diagonal Hessian.
softplus = lambda t: np.logaddexp(0.0, t)
sigma = lambda t: 1.0 / (1.0 + np.exp(-t))
ts = np.array([-6.0, -1.0, 0.0, 0.5, 3.0, 8.0])
h = 1e-5
d1 = (softplus(ts + h) - softplus(ts - h)) / (2 * h)
d2 = (softplus(ts + h) - 2 * softplus(ts) + softplus(ts - h)) / h**2
report("L2.11  f' vs sigmoid", d1, sigma(ts), tol=1e-8)
report("L2.11  f'' vs sigmoid(1 - sigmoid)", d2, sigma(ts) * (1 - sigma(ts)), tol=1e-5)
print("t        :", ts)
print("f''(t)   :", sigma(ts) * (1 - sigma(ts)))
assert (sigma(ts) * (1 - sigma(ts)) > 0).all()

# S(x) = sum softplus(x_i): the Hessian is diagonal and positive definite.
n = 5
x0 = rng.normal(size=n) * 3.0
Ssum = lambda v: softplus(v).sum()
hh = 1e-4
H_fd = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        ei, ej = hh * np.eye(n)[i], hh * np.eye(n)[j]
        H_fd[i, j] = (Ssum(x0 + ei + ej) - Ssum(x0 + ei - ej) - Ssum(x0 - ei + ej) + Ssum(x0 - ei - ej)) / (4 * hh**2)
H = np.diag(sigma(x0) * (1 - sigma(x0)))
report("L2.11  Hessian of the sum vs diag(sigma(1-sigma))", H_fd, H, tol=1e-4)
print(f"\nlambda_min(H) = {np.linalg.eigvalsh(H).min():.6e} > 0: softplus is strictly convex, ReLU is not.")

L2.11  f' vs sigmoid                                 residual = 5.504e-11
L2.11  f'' vs sigmoid(1 - sigmoid)                   residual = 2.270e-06
t        : [-6.  -1.   0.   0.5  3.   8. ]
f''(t)   : [0.0025 0.1966 0.25   0.235  0.0452 0.0003]
L2.11  Hessian of the sum vs diag(sigma(1-sigma))    residual = 3.920e-08

lambda_min(H) = 2.369216e-02 > 0: softplus is strictly convex, ReLU is not.


---

### Problem L2.12 — Curvature of Monge Surface Patches ($z = f(x,y)$)
**Source:** Spivak, *A Comprehensive Introduction to Differential Geometry* (Vol 1).  
**Problem Statement:**  
Compute the Gaussian curvature $K$ at the origin $(0,0)$ for the elliptic paraboloid $z = ax^2 + by^2$ and hyperbolic paraboloid $z = ax^2 - by^2$ ($a, b \gt 0$).

#### First-Principles Intuition
Gaussian curvature $K = \kappa_1 \kappa_2$ is the product of principal curvatures. For Monge patches $z = f(x,y)$ at critical points ($\nabla f = \mathbf{0}$), $K = \det(H_f(0,0))$.

#### Step-by-Step Solution
1. **Elliptic Paraboloid $z = ax^2 + by^2$:**
   - Gradient: $\nabla f = (2ax, 2by)^T \implies \nabla f(0,0) = \mathbf{0}$.
   - Hessian:

$$
H = \begin{bmatrix} 2a & 0 \\ 0 & 2b \end{bmatrix}
$$

   - Curvature:

$$
     K = \frac{4ab}{(1 + 0)^2} = 4ab \gt 0 \quad (\text{Elliptic geometry, bowl-shaped})
$$

2. **Hyperbolic Paraboloid $z = ax^2 - by^2$:**
   - Gradient: $\nabla f = (2ax, -2by)^T \implies \nabla f(0,0) = \mathbf{0}$.
   - Hessian:

$$
H = \begin{bmatrix} 2a & 0 \\ 0 & -2b \end{bmatrix}
$$

   - Curvature:

$$
     K = \frac{-4ab}{(1 + 0)^2} = -4ab \lt 0 \quad (\text{Hyperbolic geometry, saddle-shaped})
$$

$$
\boxed{K_{\text{elliptic}} = 4ab \gt 0, \quad K_{\text{hyperbolic}} = -4ab \lt 0}
$$

#### Key Insight / Takeaway
The determinant of the Hessian determines the sign of Gaussian intrinsic surface curvature: positive for bowls/domes, negative for saddles.

**Verification.** The cell below recomputes the boxed answer of Problem L2.12 and asserts agreement.

In [31]:
# Gaussian curvature at the origin of the two paraboloids.
x, y, a, b = sp.symbols("x y a b", positive=True)
for expr, name, sign in [(a * x**2 + b * y**2, "elliptic", +1), (a * x**2 - b * y**2, "hyperbolic", -1)]:
    Hs = sp.hessian(expr, (x, y))
    gx, gy = sp.diff(expr, x), sp.diff(expr, y)
    K = sp.simplify(Hs.det() / (1 + gx**2 + gy**2) ** 2).subs({x: 0, y: 0})
    print(f"{name:11s}: H(0,0) = {Hs.subs({x:0, y:0}).tolist()},  K(0,0) = {sp.simplify(K)}")
    assert sp.simplify(K - sign * 4 * a * b) == 0

# Numerical cross-check against the general Monge-patch formula away from the origin too.
av, bv = 0.8, 1.9
def K_monge(f, fx, fy, fxx, fxy, fyy, xx, yy):
    return (fxx(xx, yy) * fyy(xx, yy) - fxy(xx, yy) ** 2) / (1 + fx(xx, yy) ** 2 + fy(xx, yy) ** 2) ** 2
K0 = K_monge(None, lambda u, v: 2 * av * u, lambda u, v: 2 * bv * v,
             lambda u, v: 2 * av, lambda u, v: 0.0, lambda u, v: 2 * bv, 0.0, 0.0)
report("L2.12  elliptic K(0,0) vs 4ab", K0, 4 * av * bv, tol=1e-12)
K1 = K_monge(None, lambda u, v: 2 * av * u, lambda u, v: -2 * bv * v,
             lambda u, v: 2 * av, lambda u, v: 0.0, lambda u, v: -2 * bv, 0.0, 0.0)
report("L2.12  hyperbolic K(0,0) vs -4ab", K1, -4 * av * bv, tol=1e-12)
print(f"\nK_elliptic = {K0:+.4f} > 0,  K_hyperbolic = {K1:+.4f} < 0 with a = {av}, b = {bv}.")

elliptic   : H(0,0) = [[2*a, 0], [0, 2*b]],  K(0,0) = 4*a*b
hyperbolic : H(0,0) = [[2*a, 0], [0, -2*b]],  K(0,0) = -4*a*b
L2.12  elliptic K(0,0) vs 4ab                        residual = 0.000e+00
L2.12  hyperbolic K(0,0) vs -4ab                     residual = 0.000e+00

K_elliptic = +6.0800 > 0,  K_hyperbolic = -6.0800 < 0 with a = 0.8, b = 1.9.


---

## L3 — Challenge Proofs

### Problem L3.1 — Strongly Convex Functions & Global Minimum Bound
**Source:** Boyd & Vandenberghe, *Convex Optimization* (Ch. 9) / Cambridge Tripos.  
**Problem Statement:**  
Let $f: \mathbb{R}^n \to \mathbb{R}$ be $C^2$. Suppose there exists $m \gt 0$ such that $H_f(x) \succeq m I$ for all $x \in \mathbb{R}^n$. Prove that $f$ has a unique global minimizer $x^\ast$, and establish the distance bound:

$$
\lVert x - x^\ast\rVert  \le \frac{1}{m} \lVert \nabla f(x)\rVert 
$$

#### First-Principles Intuition
Strong convexity implies that $f(x)$ grows at least quadratically in all directions. This prevents flat directions at infinity and bounds distance to the minimum by gradient magnitude.

#### Step-by-Step Solution
1. **Strong Convexity Quadratic Lower Bound:**
   By Taylor's theorem with remainder, for any $x, y \in \mathbb{R}^n$, there exists $\theta \in (0, 1)$ such that:

$$
   f(y) = f(x) + \nabla f(x)^T (y - x) + \frac{1}{2} (y - x)^T H_f(x + \theta(y - x)) (y - x)
$$

   Using $H_f \succeq m I$:

$$
   f(y) \ge f(x) + \nabla f(x)^T (y - x) + \frac{m}{2} \lVert y - x\rVert^2
$$

2. **Minimize Lower Bound over $y$:**
   The right-hand side is a convex quadratic function of $y$. Setting its gradient with respect to $y$ to zero:

$$
   \nabla f(x) + m (y - x) = \mathbf{0} \implies y^\ast = x - \frac{1}{m} \nabla f(x)
$$

   Substituting $y^\ast$ into the inequality:

$$
   f(y) \ge f(x) - \frac{1}{m} \lVert \nabla f(x)\rVert^2 + \frac{m}{2} \left( \frac{1}{m^2} \lVert \nabla f(x)\rVert^2 \right) = f(x) - \frac{1}{2m} \lVert \nabla f(x)\rVert^2
$$

   Since this holds for all $y$, it holds for the minimizer $y = x^\ast$:

$$
   f(x^\ast) \ge f(x) - \frac{1}{2m} \lVert \nabla f(x)\rVert^2 \implies f(x) - f(x^\ast) \le \frac{1}{2m} \lVert \nabla f(x)\rVert^2
$$

3. **Existence and Uniqueness of $x^\ast$:**
   Fix any $x_0$. Step 1 with $x = x_0$ gives $f(y) \ge f(x_0) - \lVert \nabla f(x_0) \rVert \lVert y - x_0 \rVert + \frac{m}{2}\lVert y - x_0 \rVert^2$, whose right-hand side $\to +\infty$ as $\lVert y \rVert \to \infty$. Hence the sublevel set $\lbrace y : f(y) \le f(x_0) \rbrace$ is bounded; it is closed because $f$ is continuous, so it is compact, and a continuous $f$ attains a minimum on it. That minimiser $x^\ast$ minimises $f$ globally and satisfies $\nabla f(x^\ast) = \mathbf{0}$.

   Uniqueness: if $\nabla f(x_1) = \nabla f(x_2) = \mathbf{0}$ with $x_1 \ne x_2$, apply the step-1 inequality once in each direction and add:

$$
   f(x_2) \ge f(x_1) + \tfrac{m}{2}\lVert x_2 - x_1 \rVert^2, \qquad f(x_1) \ge f(x_2) + \tfrac{m}{2}\lVert x_2 - x_1 \rVert^2
$$

$$
   \implies 0 \ge m \lVert x_2 - x_1 \rVert^2 \implies x_1 = x_2 ,
$$

   contradicting $x_1 \ne x_2$. So the critical point, hence the global minimiser, is unique.

4. **Derive Distance Bound $\lVert x - x^\ast\rVert$:**
   Apply the strong convexity inequality setting $y = x^\ast$ and using $\nabla f(x^\ast) = \mathbf{0}$:

$$
   f(x) \ge f(x^\ast) + \nabla f(x^\ast)^T (x - x^\ast) + \frac{m}{2} \lVert x - x^\ast\rVert^2 = f(x^\ast) + \frac{m}{2} \lVert x - x^\ast\rVert^2
$$

   Combining the two inequalities:

$$
   \frac{m}{2} \lVert x - x^\ast\rVert^2 \le f(x) - f(x^\ast) \le \frac{1}{2m} \lVert \nabla f(x)\rVert^2
$$

$$
   m^2 \lVert x - x^\ast\rVert^2 \le \lVert \nabla f(x)\rVert^2 \implies \lVert x - x^\ast\rVert  \le \frac{1}{m} \lVert \nabla f(x)\rVert 
$$

$$
\boxed{\lVert x - x^\ast\rVert  \le \frac{1}{m} \lVert \nabla f(x)\rVert}
$$

#### Key Insight / Takeaway
Strong convexity guarantees that a small gradient $\lVert \nabla f(x)\rVert  \le \epsilon$ places the current iterate within distance $\frac{\epsilon}{m}$ of the global minimizer.

**Verification.** The cell below recomputes the boxed answer of Problem L3.1 and asserts agreement.

In [32]:
# Strong convexity: the bound ||x - x*|| <= ||grad f(x)|| / m, on a function that is
# strongly convex but not quadratic.
n = 4
Q = np.linalg.qr(rng.normal(size=(n, n)))[0]
lam = np.array([1.0, 2.0, 3.5, 6.0])
A = Q @ np.diag(lam) @ Q.T
A = 0.5 * (A + A.T)
c = rng.normal(size=n)
f = lambda v: 0.5 * v @ A @ v + np.sum(np.log(np.cosh(v - c)))   # Hessian = A + diag(sech^2) >= A
grad = lambda v: A @ v + np.tanh(v - c)
m_ = lam.min()                                   # A >= m I and the log-cosh part adds >= 0

from scipy.optimize import minimize
res = minimize(f, np.zeros(n), jac=grad, method="BFGS", tol=1e-14)
x_star = res.x
report("L3.1  gradient at the minimiser", grad(x_star), np.zeros(n), tol=1e-7)

print(f"m = {m_:.4f}     x* = {x_star}")
print("\n   ||x - x*||        ||grad f(x)|| / m      slack")
worst = -np.inf
for x0 in rng.normal(size=(2000, n)) * 3.0:
    lhs = np.linalg.norm(x0 - x_star)
    rhs = np.linalg.norm(grad(x0)) / m_
    worst = max(worst, lhs - rhs)
    assert lhs <= rhs + 1e-9
for x0 in rng.normal(size=(4, n)) * 3.0:
    lhs, rhs = np.linalg.norm(x0 - x_star), np.linalg.norm(grad(x0)) / m_
    print(f"   {lhs:12.6f}     {rhs:16.6f}   {rhs - lhs:12.6f}")
print(f"\nworst violation over 2000 random x: {worst:.3e} (never positive)")

# Uniqueness: a second BFGS run from a far start finds the same point.
res2 = minimize(f, 50 * rng.normal(size=n), jac=grad, method="BFGS", tol=1e-14)
report("L3.1  minimiser from a far start", res2.x, x_star, tol=1e-6)

L3.1  gradient at the minimiser                      residual = 8.188e-16
m = 1.0000     x* = [-0.0022 -0.1804  0.1564 -0.3627]

   ||x - x*||        ||grad f(x)|| / m      slack
       4.079539            10.233615       6.154076
       5.350099            24.807293      19.457193
       7.894974            39.714313      31.819339
       2.217620             9.025339       6.807718

worst violation over 2000 random x: -1.043e+00 (never positive)
L3.1  minimiser from a far start                     residual = 7.189e-15


---

### Problem L3.2 — Newton's Method Stays in Its Quadratic-Convergence Basin
**Source:** Nocedal & Wright, *Numerical Optimization* (Ch. 3).
**Problem Statement:**
Derivation 3.5 of `first_principles.ipynb` establishes the one-step bound $\lVert e_{k+1}\rVert  \le \frac{ML}{2}\lVert e_k\rVert^2$ for Newton's method under $L$-Lipschitz $H_f$ and $\lVert H_f^{-1}(x)\rVert  \le M$ near $x^\ast$. That bound alone does not prove convergence: it says nothing about whether $x^{(k+1)}$ stays inside the neighborhood where the Lipschitz/boundedness hypotheses hold. Prove that if $\lVert e_0\rVert  \lt r := \min\left(\frac{1}{ML}, \rho\right)$ (where $\rho$ is the radius of the neighborhood on which the hypotheses hold), then every iterate $x^{(k)}$ remains in that neighborhood and $e_k \to 0$.

#### First-Principles Intuition
A one-step bound only controls the *next* error relative to the *current* one; without confining the iterates to the region where $M$ and $L$ are valid, nothing stops the sequence from leaving that region after a few steps and the bound becoming meaningless. Closing this gap needs an induction that keeps the iterate trapped.

#### Step-by-Step Solution
Let $c = ML$ and define $t_k = c\lVert e_k\rVert$. The one-step bound reads $\lVert e_{k+1}\rVert  \le \frac{c}{2}\lVert e_k\rVert^2$, i.e. $t_{k+1} \le \frac{1}{2}t_k^2$.

**Claim:** if $t_0 \lt 1$ (equivalently $\lVert e_0\rVert  \lt 1/c = 1/(ML)$), then $t_k \to 0$ monotonically and $t_k \le t_0$ for all $k$, so $\lVert e_k\rVert  \le \lVert e_0\rVert  \lt r \le \rho$ for every $k$ — the iterates never leave the neighborhood.

**Induction.** Base case $k=0$ holds by hypothesis. Suppose $t_k \le t_0 \lt 1$. Then

$$
t_{k+1} \le \tfrac{1}{2}t_k^2 = t_k \cdot \tfrac{1}{2}t_k \le t_k \cdot \tfrac{1}{2}t_0 \lt t_k \le t_0,
$$

using $t_0 \lt 1 \Rightarrow \tfrac12 t_0 \lt \tfrac12 \lt 1$. So $t_{k+1} \lt t_k \le t_0$, completing the induction: the sequence $(t_k)$ is strictly decreasing (once $k\ge1$) and bounded below by $0$, hence converges to some $t_\infty \ge 0$ satisfying $t_\infty \le \frac12 t_\infty^2$, whose only solution with $t_\infty \lt 1$ is $t_\infty = 0$.

Since $\lVert e_k\rVert  = t_k / c \le t_0/c = \lVert e_0\rVert  \lt r \le \rho$ for every $k$, each $x^{(k)}$ lies in the neighborhood where the Lipschitz and boundedness hypotheses hold, so the one-step bound applies at every iteration — the induction is self-consistent. Unwinding $t_{k+1} \le \frac12 t_k^2$ also gives the closed form $t_k \le (2t_0)^{2^k}/2$, i.e.

$$
\boxed{\lVert e_k\rVert  \le \frac{1}{ML}\cdot\frac{(ML\lVert e_0\rVert \cdot 2)^{2^k}}{2} \longrightarrow 0 \text{ super-exponentially, provided } \lVert e_0\rVert  \lt \min\!\left(\frac{1}{ML}, \rho\right)}
$$

#### Key Insight / Takeaway
Quadratic-convergence bounds are only as strong as the invariant that keeps the iterates inside their region of validity; the basin radius $1/(ML)$ falls directly out of requiring $t_k$ to be a contraction, not just decreasing in the limit.

**Verification.** The cell below recomputes the boxed answer of Problem L3.2 and asserts agreement.

In [33]:
# The basin argument: t_k = ML||e_k|| obeys t_{k+1} <= t_k^2 / 2, so t_0 < 1 traps the
# iterates and drives them to zero super-exponentially.
def orbit(t0, steps=6):
    ts, t = [t0], t0
    for _ in range(steps):
        t = 0.5 * t * t
        ts.append(t)
    return np.array(ts)

for t0 in (0.9, 0.5, 0.1):
    ts = orbit(t0)
    bound = np.array([(2 * t0) ** (2 ** k) / 2 for k in range(len(ts))])
    print(f"t0 = {t0}:  t_k = {ts}")
    assert (np.diff(ts) < 0).all() and (ts <= t0 + 1e-15).all() and ts[-1] < 1e-12
    assert (ts <= bound + 1e-15).all()
print("\nEvery orbit with t0 < 1 decreases monotonically and stays below t0: the basin is invariant.")

# t0 > 2 escapes: the same recursion blows up, so the radius condition is not decorative.
ts_bad = orbit(2.5, 4)
print(f"t0 = 2.5: t_k = {ts_bad}  -> diverges")
assert ts_bad[-1] > ts_bad[0]

# A real Newton run on f = x^4 + y^4 - 4xy, whose minimiser is (1,1).
f = lambda v: v[0]**4 + v[1]**4 - 4 * v[0] * v[1]
grad = lambda v: np.array([4 * v[0]**3 - 4 * v[1], 4 * v[1]**3 - 4 * v[0]])
hess = lambda v: np.array([[12 * v[0]**2, -4.0], [-4.0, 12 * v[1]**2]])
x, errs = np.array([1.6, 1.4]), []
for _ in range(6):
    errs.append(np.linalg.norm(x - np.array([1.0, 1.0])))
    x = x - np.linalg.solve(hess(x), grad(x))
errs = np.array(errs)
print("\nNewton errors:", errs)
assert (np.diff(errs) < 0).all()
ratios = errs[1:] / errs[:-1] ** 2
print("e_{k+1}/e_k^2 :", ratios[errs[:-1] > 1e-8])
assert np.isfinite(ratios[0]) and ratios[0] < 10

t0 = 0.9:  t_k = [0.9    0.405  0.082  0.0034 0.     0.     0.    ]
t0 = 0.5:  t_k = [0.5    0.125  0.0078 0.     0.     0.     0.    ]
t0 = 0.1:  t_k = [0.1   0.005 0.    0.    0.    0.    0.   ]

Every orbit with t0 < 1 decreases monotonically and stays below t0: the basin is invariant.
t0 = 2.5: t_k = [ 2.5     3.125   4.8828 11.9209 71.0543]  -> diverges

Newton errors: [0.7211 0.2566 0.0501 0.0025 0.     0.    ]
e_{k+1}/e_k^2 : [0.4935 0.76   0.9997 1.0767 1.0797]


---

### Problem L3.3 — Courant-Fischer Min-Max Theorem for Rayleigh Quotient
**Source:** Horn & Johnson, *Matrix Analysis* (Ch. 4) / Putnam Archive.  
**Problem Statement:**  
Let $A \in \mathbb{R}^{n \times n}$ be a real symmetric matrix with eigenvalues $\lambda_1 \le \lambda_2 \le \dots \le \lambda_n$. Prove the Courant-Fischer Min-Max characterization for the $k$-th eigenvalue:

$$
\lambda_k = \min_{\dim(S) = k} \max_{x \in S, x \neq \mathbf{0}} \frac{x^T A x}{x^T x}
$$

where $S$ ranges over all $k$-dimensional subspaces of $\mathbb{R}^n$.

#### First-Principles Intuition
The $k$-th eigenvalue is the maximum Rayleigh quotient achievable within the best possible $k$-dimensional subspace, balancing subspace expansion against curvature bounds.

#### Step-by-Step Solution
Let $\{u_1, u_2, \dots, u_n\}$ be an orthonormal basis of eigenvectors of $A$, with $A u_i = \lambda_i u_i$.

1. **Upper Bound Construction:**
   Choose specific subspace $S_0 = \operatorname{span}(u_1, u_2, \dots, u_k)$, which has dimension $k$.  
   Any $x \in S_0$ can be written as $x = \sum_{i=1}^k c_i u_i$.  
   Then:

$$
   \frac{x^T A x}{x^T x} = \frac{\sum_{i=1}^k \lambda_i c_i^2}{\sum_{i=1}^k c_i^2} \le \frac{\lambda_k \sum_{i=1}^k c_i^2}{\sum_{i=1}^k c_i^2} = \lambda_k
$$

   Thus $\max_{x \in S_0, x \neq \mathbf{0}} R_A(x) = \lambda_k$.  
   Taking the minimum over all $k$-dimensional subspaces $S$:

$$
   \min_{\dim(S) = k} \max_{x \in S, x \neq \mathbf{0}} R_A(x) \le \max_{x \in S_0, x \neq \mathbf{0}} R_A(x) = \lambda_k
$$

2. **Lower Bound Construction:**
   Let $S$ be any arbitrary $k$-dimensional subspace. Define another subspace $V = \operatorname{span}(u_k, u_{k+1}, \dots, u_n)$, which has dimension $n - k + 1$.  
   By the dimension theorem for vector spaces:

$$
   \dim(S \cap V) = \dim(S) + \dim(V) - \dim(S + V) \ge k + (n - k + 1) - n = 1
$$

   Thus, there exists a non-zero vector $x_0 \in S \cap V$.  
   Since $x_0 \in V$, $x_0 = \sum_{i=k}^n d_i u_i$, so:

$$
   R_A(x_0) = \frac{\sum_{i=k}^n \lambda_i d_i^2}{\sum_{i=k}^n d_i^2} \ge \frac{\lambda_k \sum_{i=k}^n d_i^2}{\sum_{i=k}^n d_i^2} = \lambda_k
$$

   Since $x_0 \in S$, $\max_{x \in S, x \neq \mathbf{0}} R_A(x) \ge R_A(x_0) \ge \lambda_k$.  
   Since this holds for every $k$-dimensional subspace $S$:

$$
   \min_{\dim(S) = k} \max_{x \in S, x \neq \mathbf{0}} R_A(x) \ge \lambda_k
$$

Combining both inequalities yields the exact equality:

$$
\boxed{\lambda_k = \min_{\dim(S) = k} \max_{x \in S, x \neq \mathbf{0}} \frac{x^T A x}{x^T x}}
$$

#### Key Insight / Takeaway
The Courant-Fischer Min-Max Theorem provides a variational definition of intermediate matrix eigenvalues without requiring explicit characteristic polynomials.

**Verification.** The cell below recomputes the boxed answer of Problem L3.3 and asserts agreement.

In [34]:
# Courant-Fischer, checked by brute force on a small symmetric matrix.
n = 5
Q = np.linalg.qr(rng.normal(size=(n, n)))[0]
lam_true = np.array([-2.0, -0.3, 0.8, 2.5, 6.0])          # ascending
A = Q @ np.diag(lam_true) @ Q.T
A = 0.5 * (A + A.T)
lam = np.linalg.eigvalsh(A)                                # ascending
U = np.linalg.eigh(A)[1]

def max_R_on(S):
    """max Rayleigh quotient over the column span of S: the top eigenvalue of S^T A S in the S^T S metric."""
    Sq = np.linalg.qr(S)[0]
    return np.linalg.eigvalsh(Sq.T @ A @ Sq).max()

for k in range(1, n + 1):
    S0 = U[:, :k]                                          # the optimal subspace span(u_1..u_k)
    best = max_R_on(S0)
    report(f"L3.3  max R on span(u_1..u_{k}) vs lambda_{k}", best, lam[k - 1], tol=1e-12)
    worst = min(max_R_on(rng.normal(size=(n, k))) for _ in range(3000))
    print(f"  k = {k}: optimal subspace gives {best:+.6f}, best of 3000 random k-subspaces {worst:+.6f}")
    assert worst >= lam[k - 1] - 1e-9
print("\nNo k-dimensional subspace beats span(u_1..u_k): the min over subspaces is lambda_k.")

L3.3  max R on span(u_1..u_1) vs lambda_1            residual = 1.554e-15
  k = 1: optimal subspace gives -2.000000, best of 3000 random k-subspaces -1.847282
L3.3  max R on span(u_1..u_2) vs lambda_2            residual = 1.110e-16
  k = 2: optimal subspace gives -0.300000, best of 3000 random k-subspaces -0.180431
L3.3  max R on span(u_1..u_3) vs lambda_3            residual = 2.220e-16


  k = 3: optimal subspace gives +0.800000, best of 3000 random k-subspaces +0.824904
L3.3  max R on span(u_1..u_4) vs lambda_4            residual = 8.882e-16
  k = 4: optimal subspace gives +2.500000, best of 3000 random k-subspaces +2.500000
L3.3  max R on span(u_1..u_5) vs lambda_5            residual = 2.665e-15


  k = 5: optimal subspace gives +6.000000, best of 3000 random k-subspaces +6.000000

No k-dimensional subspace beats span(u_1..u_k): the min over subspaces is lambda_k.


---

### Problem L3.4 — BFGS Secant Equation & Positive Definiteness Preservation
**Source:** Nocedal & Wright, *Numerical Optimization* (Ch. 6).  
**Problem Statement:**  
The BFGS Quasi-Newton update matrix $H_{k+1}$ approximating the inverse Hessian $H^{-1}$ is:

$$
H_{k+1} = (I - \rho_k s_k y_k^T) H_k (I - \rho_k y_k s_k^T) + \rho_k s_k s_k^T, \quad \text{where } \rho_k = \frac{1}{y_k^T s_k}
$$

Since $H_k$ approximates the *inverse* Hessian, its secant equation is written in $H$-form, $H_{k+1} y_k = s_k$ — the dual of the $B$-form secant equation $B_{k+1} s_k = y_k$ satisfied by an approximation $B_k$ of the Hessian itself. Show that $H_{k+1} y_k = s_k$, and prove that if $H_k \succ 0$, then $H_{k+1} \succ 0$ if and only if the curvature condition $y_k^T s_k \gt 0$ holds.

#### First-Principles Intuition
The BFGS update builds a symmetric second-order approximation satisfying the secant equation $H_{k+1} y_k = s_k$ (equivalently $B_{k+1} s_k = y_k$ for the direct Hessian approximation $B_k = H_k^{-1}$). The curvature condition $y_k^T s_k \gt 0$ ensures strictly positive inner products along update steps.

#### Step-by-Step Solution
1. **Verify Secant Equation ($H_{k+1} y_k = s_k$):**
   Multiply $H_{k+1}$ by $y_k$:

$$
   H_{k+1} y_k = (I - \rho_k s_k y_k^T) H_k (I - \rho_k y_k s_k^T) y_k + \rho_k s_k s_k^T y_k
$$

   Notice $(I - \rho_k y_k s_k^T) y_k = y_k - \rho_k y_k (s_k^T y_k) = y_k - \frac{y_k s_k^T y_k}{y_k^T s_k} = \mathbf{0}$.  
   Thus the first term vanishes!  
   The second term is $\rho_k s_k (s_k^T y_k) = \frac{s_k (y_k^T s_k)}{y_k^T s_k} = s_k$.  
   Therefore, $H_{k+1} y_k = s_k$.

2. **Proof of Positive Definiteness Preservation:**
   Let $z \neq \mathbf{0}$. Compute $z^T H_{k+1} z$:

$$
   z^T H_{k+1} z = z^T (I - \rho_k s_k y_k^T) H_k (I - \rho_k y_k s_k^T) z + \rho_k (s_k^T z)^2
$$

   Let $v = (I - \rho_k y_k s_k^T) z$. Then $z^T H_{k+1} z = v^T H_k v + \rho_k (s_k^T z)^2$.

   - If $v = \mathbf{0}$, then $z = \rho_k (s_k^T z) y_k$, so $z$ is a scalar multiple of $y_k$. If that scalar $\rho_k(s_k^Tz)$ were $0$ then $z = \mathbf{0}$, contradicting $z \ne \mathbf{0}$; hence $s_k^T z \ne 0$ and
$z^T H_{k+1} z = 0 + \rho_k (s_k^T z)^2 \gt 0$, since $(s_k^Tz)^2 \gt 0$ and $\rho_k = 1/(y_k^Ts_k)$ is positive exactly when the curvature condition $y_k^T s_k \gt 0$ holds — giving that direction of the "iff".
   - If $v \neq \mathbf{0}$, $v^T H_k v \gt 0$. Since $\rho_k (s_k^T z)^2 \ge 0$, $z^T H_{k+1} z \gt 0$.

$$
\boxed{H_{k+1} y_k = s_k \quad \text{and} \quad H_k \succ 0, \, y_k^T s_k \gt 0 \iff H_{k+1} \succ 0}
$$

#### Key Insight / Takeaway
The condition $y_k^T s_k \gt 0$ guarantees that BFGS updates preserve positive definite Hessians, maintaining descent directions without explicit matrix inversion.

**Verification.** The cell below recomputes the boxed answer of Problem L3.4 and asserts agreement.

In [35]:
# BFGS: the secant equation, and positive definiteness exactly under y^T s > 0.
def bfgs_update(H, s, y):
    rho = 1.0 / (y @ s)
    I = np.eye(H.shape[0])
    return (I - rho * np.outer(s, y)) @ H @ (I - rho * np.outer(y, s)) + rho * np.outer(s, s)

n = 6
B = rng.normal(size=(n, n))
H = B @ B.T + n * np.eye(n)
for _ in range(5):
    s = rng.normal(size=n)
    G = rng.normal(size=(n, n)); G = G @ G.T + np.eye(n)     # a positive definite "true Hessian"
    y = G @ s                                                # then y^T s = s^T G s > 0
    assert y @ s > 0
    Hn = bfgs_update(H, s, y)
    report("L3.4  secant equation H_{k+1} y = s", Hn @ y, s, tol=1e-9)
    report("L3.4  symmetry of H_{k+1}", Hn, Hn.T, tol=1e-12)
    assert np.linalg.eigvalsh(Hn).min() > 0
print(f"lambda_min(H_+) stayed positive on all five updates.")

# Violate the curvature condition and positive definiteness is lost.
s = rng.normal(size=n)
y = -0.4 * s + 0.05 * rng.normal(size=n)
print(f"\ncurvature condition y^T s = {y @ s:+.6f} < 0")
Hbad = bfgs_update(H, s, y)
lam_bad = np.linalg.eigvalsh(Hbad)
print("eigenvalues of H_+ :", lam_bad)
report("L3.4  secant equation still holds", Hbad @ y, s, tol=1e-9)
assert lam_bad.min() < 0
print("The secant equation survives, positive definiteness does not: y^T s > 0 is the exact hinge.")

L3.4  secant equation H_{k+1} y = s                  residual = 6.911e-15
L3.4  symmetry of H_{k+1}                            residual = 1.776e-15
L3.4  secant equation H_{k+1} y = s                  residual = 9.853e-15
L3.4  symmetry of H_{k+1}                            residual = 1.776e-15
L3.4  secant equation H_{k+1} y = s                  residual = 3.619e-14
L3.4  symmetry of H_{k+1}                            residual = 8.882e-16
L3.4  secant equation H_{k+1} y = s                  residual = 4.585e-14
L3.4  symmetry of H_{k+1}                            residual = 2.665e-15
L3.4  secant equation H_{k+1} y = s                  residual = 3.664e-14
L3.4  symmetry of H_{k+1}                            residual = 8.882e-16
lambda_min(H_+) stayed positive on all five updates.

curvature condition y^T s = -0.942728 < 0
eigenvalues of H_+ : [-2.1524  6.498   7.3988  9.0687 14.339  23.4434]
L3.4  secant equation still holds                    residual = 6.661e-16
The secant equation

---

### Problem L3.5 — Infinite-Dimensional Functional Jacobian Determinant
**Source:** Gelfand & Fomin, *Calculus of Variations* / Cambridge Tripos.  
**Problem Statement:**  
Consider a linear change of function variables in path integration $y(t) = x(t) + \int_0^t K(t, s) x(s) \, ds$ for $t \in [0, T]$, where $K(t, s)$ is a continuous Volterra kernel. Prove that the functional Jacobian determinant $\det(J) = 1$.

#### First-Principles Intuition
A Volterra integral operator represents a lower-triangular linear transformation in discretized time. Since all diagonal entries of a lower-triangular matrix with unit main diagonal are 1, its determinant is identically 1.

#### Step-by-Step Solution
Discretize the interval $[0, T]$ into $N$ equal steps $t_i = i \Delta t$ with $\Delta t = \frac{T}{N}$.  
The transformation equation for $y_i = y(t_i)$ is:

$$
y_i = x_i + \sum_{j=1}^{i-1} K(t_i, t_j) x_j \Delta t
$$

Construct the $N \times N$ discrete Jacobian matrix $J_{ij} = \frac{\partial y_i}{\partial x_j}$:

$$
J_{ij} = \begin{cases}
1 & \text{if } i = j \\
0 & \text{if } j \gt i \quad (\text{upper triangle}) \\
K(t_i, t_j) \Delta t & \text{if } j \lt i \quad (\text{strictly lower triangle})
\end{cases}
$$

For each finite $N$, $J$ is unit lower-triangular:

$$
J = \begin{bmatrix}
1 & 0 & 0 & \cdots & 0 \\
K_{21} \Delta t & 1 & 0 & \cdots & 0 \\
K_{31} \Delta t & K_{32} \Delta t & 1 & \cdots & 0 \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
K_{N1} \Delta t & K_{N2} \Delta t & K_{N3} \Delta t & \cdots & 1
\end{bmatrix}
$$

The determinant of a triangular matrix is the product of its diagonal elements:

$$
\det(J_N) = \prod_{i=1}^N 1 = 1 \quad \text{for every finite } N.
$$

This alone does not define the functional determinant $\det(J)$ of the continuum operator — a limit of finite determinants is not automatically the determinant of the limiting operator. The rigorous object is the **Fredholm determinant** of $J = I + K$ for the Volterra integral operator $K$, defined by $\det(I+K) = \exp(\operatorname{tr}\log(I+K)) = \exp\left(-\sum_{n=1}^\infty \frac{(-1)^n}{n}\operatorname{tr}(K^n)\right)$. A Volterra kernel is *quasi-nilpotent*: $\operatorname{tr}(K^n) = \int_0^T \!\!\cdots\!\! \int_0^T K(t_1,t_2)K(t_2,t_3)\cdots K(t_n,t_1)\,dt_1\cdots dt_n = 0$ for every $n \ge 1$, because the integrand is supported only where $t_1 \ge t_2 \ge \cdots \ge t_n \ge t_1$ simultaneously — a measure-zero set unless all $t_i$ coincide. Hence every term in the trace-log series vanishes, and

$$
\boxed{\det(I+K) = \exp(0) = 1 \implies \mathcal{D}y = \mathcal{D}x \quad (\text{measure-preserving functional transformation})}
$$

which is the rigorous version of the same conclusion the finite-$N$ computation suggested.

#### Key Insight / Takeaway
The finite-dimensional discretization is only heuristic evidence; the actual proof needs the Fredholm-determinant definition, and it works because a Volterra kernel's iterated traces vanish identically — the continuum analogue of "the diagonal of a strictly lower-triangular matrix is zero."

**Verification.** The cell below recomputes the boxed answer of Problem L3.5 and asserts agreement.

In [36]:
# Volterra kernel: the discretised Jacobian is unit lower-triangular with determinant 1,
# and every iterated trace tr(K^n) vanishes, which is what the Fredholm argument needs.
T = 1.0
K = lambda t, s: np.exp(-(t - s)) * (1.0 + 3.0 * np.sin(4.0 * t * s))   # any continuous kernel
for N in (10, 50, 200, 800):
    dt = T / N
    tg = (np.arange(N) + 0.5) * dt
    Kd = np.tril(K(tg[:, None], tg[None, :]), -1) * dt
    J = np.eye(N) + Kd
    sign, logabsdet = np.linalg.slogdet(J)
    print(f"  N = {N:4d}:  det J = {sign * np.exp(logabsdet):.15f}   "
          f"tr(K) = {np.trace(Kd):.3e}   tr(K^2) = {np.trace(Kd @ Kd):.3e}   tr(K^3) = {np.trace(np.linalg.matrix_power(Kd, 3)):.3e}")
    report(f"L3.5  det J at N = {N}", sign * np.exp(logabsdet), 1.0, tol=1e-12)
    assert abs(np.trace(Kd)) < 1e-14
    for p in (2, 3, 4):
        assert abs(np.trace(np.linalg.matrix_power(Kd, p))) < 1e-14
print("\nStrictly lower-triangular K is nilpotent, so every tr(K^n) is exactly 0 and")
print("det(I + K) = exp(-sum (-1)^n tr(K^n)/n) = exp(0) = 1, independently of N.")

  N =   10:  det J = 1.000000000000000   tr(K) = 0.000e+00   tr(K^2) = 0.000e+00   tr(K^3) = 0.000e+00
L3.5  det J at N = 10                                residual = 0.000e+00
  N =   50:  det J = 1.000000000000000   tr(K) = 0.000e+00   tr(K^2) = 0.000e+00   tr(K^3) = 0.000e+00
L3.5  det J at N = 50                                residual = 0.000e+00
  N =  200:  det J = 1.000000000000000   tr(K) = 0.000e+00   tr(K^2) = 0.000e+00   tr(K^3) = 0.000e+00
L3.5  det J at N = 200                               residual = 0.000e+00


  N =  800:  det J = 1.000000000000000   tr(K) = 0.000e+00   tr(K^2) = 0.000e+00   tr(K^3) = 0.000e+00
L3.5  det J at N = 800                               residual = 0.000e+00

Strictly lower-triangular K is nilpotent, so every tr(K^n) is exactly 0 and
det(I + K) = exp(-sum (-1)^n tr(K^n)/n) = exp(0) = 1, independently of N.


---

### Problem L3.6 — Escaping Saddle Points via Local Negative Curvature Trajectory
**Source:** Polyak (1987) & Ge et al. (2015, *Escaping From Saddle Points*).  
**Problem Statement:**  
Let $f(x, y) = \frac{1}{2} \lambda_1 x^2 + \frac{1}{2} \lambda_2 y^2$ with $\lambda_1 \lt 0 \lt \lambda_2$ describe a local saddle point at $(0,0)$. For perturbed gradient descent $x^{(k+1)} = x^{(k)} - \eta \nabla f(x^{(k)}) + \xi^{(k)}$ with step size $\eta \lt \frac{2}{\lambda_2}$, derive the exponential escape trajectory along the negative curvature axis $x$.

#### First-Principles Intuition
Near a saddle point, gradient descent along positive curvature directions decays to zero, while perturbations along negative curvature directions ($\lambda_1 \lt 0$) amplify exponentially.

#### Step-by-Step Solution
1. **Gradient Components:**

$$
   \nabla f(x, y) = \begin{bmatrix} \lambda_1 x \\ \lambda_2 y \end{bmatrix}
$$

2. **Unperturbed Recurrence Relations:**

$$
   x^{(k+1)} = x^{(k)} - \eta (\lambda_1 x^{(k)}) = (1 - \eta \lambda_1) x^{(k)}
$$

$$
   y^{(k+1)} = y^{(k)} - \eta (\lambda_2 y^{(k)}) = (1 - \eta \lambda_2) y^{(k)}
$$

3. **Analyze Growth Factors:**
   - Along positive curvature axis $y$ ($\lambda_2 \gt 0$): Since $\eta \lt \frac{2}{\lambda_2}$, $\lvert 1 - \eta \lambda_2 \rvert \lt 1$, so $y^{(k)} = (1 - \eta \lambda_2)^k y^{(0)} \to 0$ exponentially.
   - Along negative curvature axis $x$ ($\lambda_1 = -\lvert\lambda_1\rvert \lt 0$):

$$
     1 - \eta \lambda_1 = 1 + \eta \lvert\lambda_1\rvert \gt 1
$$

     Thus, after $k$ steps:

$$
     x^{(k)} = (1 + \eta \lvert\lambda_1\rvert)^k x^{(0)}
$$

4. **Incorporate Initial Noise Perturbation $\xi_x$:**
   Even if starting exact at $x^{(0)} = 0$, a small random perturbation $\xi_x \gt 0$ yields:

$$
   x^{(k)} = (1 + \eta \lvert\lambda_1\rvert)^k \xi_x = e^{k \ln(1 + \eta \lvert\lambda_1\rvert)} \xi_x \approx e^{k \eta \lvert\lambda_1\rvert} \xi_x
$$

   The iterate escapes the saddle point neighborhood ($\lVert x^{(k)}\rVert  \ge R$) in time:

$$
   k_{\text{escape}} \approx \frac{1}{\eta \lvert\lambda_1\rvert} \ln \left( \frac{R}{\xi_x} \right)
$$

$$
\boxed{x^{(k)} = (1 + \eta \lvert\lambda_1\rvert)^k x^{(0)} \implies \text{Exponential escape along negative curvature eigenvector with rate } 1 + \eta \lvert\lambda_1\rvert \gt 1}
$$

#### Key Insight / Takeaway
Negative Hessian eigenvalues ($\lambda_1 \lt 0$) cause gradient descent to act as an exponential amplifier for random noise, facilitating saddle point escape.

**Verification.** The cell below recomputes the boxed answer of Problem L3.6 and asserts agreement.

In [37]:
# Saddle escape: the negative-curvature coordinate grows like (1 + eta|lambda_1|)^k.
lam1, lam2, eta = -0.6, 3.0, 0.5      # eta < 2/lambda_2 = 0.667
assert eta < 2 / lam2
gx, gy = 1 - eta * lam1, 1 - eta * lam2
print(f"growth factor along x (lambda_1 = {lam1}) = {gx:.6f}   (> 1)")
print(f"decay  factor along y (lambda_2 = {lam2}) = {gy:.6f}   (|.| < 1)")
assert gx > 1 and abs(gy) < 1

xi = 1e-8                              # tiny perturbation off the saddle
xs, ys = [xi], [1.0]
for _ in range(120):
    xs.append((1 - eta * lam1) * xs[-1])
    ys.append((1 - eta * lam2) * ys[-1])
xs, ys = np.array(xs), np.array(ys)
report("L3.6  x_k vs (1 + eta|lambda_1|)^k xi", xs / xs, np.ones(len(xs)), tol=1e-9)
report("L3.6  relative error of the closed form", xs / (gx ** np.arange(len(xs)) * xi), np.ones(len(xs)), tol=1e-9)
assert abs(ys[-1]) < 1e-9

R = 1.0
hit = np.flatnonzero(np.abs(xs) >= R)
assert hit.size > 0, "extend the iteration: the orbit has not escaped yet"
k_meas = int(hit[0])
k_pred = np.log(R / xi) / np.log(gx)
print(f"\nmeasured escape time to |x| >= {R}: k = {k_meas}")
print(f"predicted ln(R/xi)/ln(1 + eta|lambda_1|) = {k_pred:.3f}")
assert abs(k_meas - k_pred) <= 1.0
print(f"the coarser estimate ln(R/xi)/(eta|lambda_1|) = {np.log(R/xi)/(eta*abs(lam1)):.3f} "
      f"is the eta|lambda_1| << 1 approximation")

growth factor along x (lambda_1 = -0.6) = 1.300000   (> 1)
decay  factor along y (lambda_2 = 3.0) = -0.500000   (|.| < 1)
L3.6  x_k vs (1 + eta|lambda_1|)^k xi                residual = 0.000e+00
L3.6  relative error of the closed form              residual = 4.441e-16

measured escape time to |x| >= 1.0: k = 71
predicted ln(R/xi)/ln(1 + eta|lambda_1|) = 70.210
the coarser estimate ln(R/xi)/(eta|lambda_1|) = 61.402 is the eta|lambda_1| << 1 approximation


---

### Problem L3.7 — First & Second Fundamental Forms of a Monge Surface Patch
**Source:** Cambridge Tripos Part II Differential Geometry / Spivak.  
**Problem Statement:**  
For a smooth Monge surface patch $r(x,y) = (x, y, f(x,y))^T$, derive the First Fundamental Form matrix:

$$
I = \begin{bmatrix} E & F \\ F & G \end{bmatrix}
$$

and Second Fundamental Form matrix:

$$
II = \begin{bmatrix} L & M \\ M & N \end{bmatrix}
$$

in terms of gradient $\nabla f$ and Hessian $H_f$.

#### First-Principles Intuition
The First Fundamental Form measures intrinsic surface metric distances ($ds^2$), while the Second Fundamental Form measures extrinsic surface bending in 3D space using normal projection of second partial derivatives.

#### Step-by-Step Solution
1. **Tangent Vectors:**

$$
   r_x = \frac{\partial r}{\partial x} = \begin{bmatrix} 1 \\ 0 \\ f_x \end{bmatrix}, \quad r_y = \frac{\partial r}{\partial y} = \begin{bmatrix} 0 \\ 1 \\ f_y \end{bmatrix}
$$

2. **First Fundamental Form Matrix $I$:**
   - $E = r_x \cdot r_x = 1 + f_x^2$
   - $F = r_x \cdot r_y = f_x f_y$
   - $G = r_y \cdot r_y = 1 + f_y^2$

$$
   I = \begin{bmatrix} 1 + f_x^2 & f_x f_y \\ f_x f_y & 1 + f_y^2 \end{bmatrix} = I_2 + \nabla f \nabla f^T
$$

3. **Unit Normal Vector $n$:**

$$
   r_x \times r_y = \begin{vmatrix} \hat{i} & \hat{j} & \hat{k} \\ 1 & 0 & f_x \\ 0 & 1 & f_y \end{vmatrix} = \begin{bmatrix} -f_x \\ -f_y \\ 1 \end{bmatrix} \implies n = \frac{1}{\sqrt{1 + f_x^2 + f_y^2}} \begin{bmatrix} -f_x \\ -f_y \\ 1 \end{bmatrix}
$$

4. **Second Derivatives of $r(x,y)$:**

$$
   r_{xx} = \begin{bmatrix} 0 \\ 0 \\ f_{xx} \end{bmatrix}, \quad r_{xy} = \begin{bmatrix} 0 \\ 0 \\ f_{xy} \end{bmatrix}, \quad r_{yy} = \begin{bmatrix} 0 \\ 0 \\ f_{yy} \end{bmatrix}
$$

5. **Second Fundamental Form Matrix $II$:**
   - $L = r_{xx} \cdot n = \frac{f_{xx}}{\sqrt{1 + \lVert \nabla f\rVert^2}}$
   - $M = r_{xy} \cdot n = \frac{f_{xy}}{\sqrt{1 + \lVert \nabla f\rVert^2}}$
   - $N = r_{yy} \cdot n = \frac{f_{yy}}{\sqrt{1 + \lVert \nabla f\rVert^2}}$

$$
   II = \frac{1}{\sqrt{1 + \lVert \nabla f\rVert^2}} \begin{bmatrix} f_{xx} & f_{xy} \\ f_{xy} & f_{yy} \end{bmatrix} = \frac{H_f}{\sqrt{1 + \lVert \nabla f\rVert^2}}
$$

$$
\boxed{I = I_2 + \nabla f \nabla f^T, \quad II = \frac{H_f}{\sqrt{1 + \lVert \nabla f\rVert^2}}}
$$

#### Key Insight / Takeaway
The Second Fundamental Form of a surface function $z = f(x,y)$ is directly proportional to its Hessian matrix $H_f$, scaled by the surface normal factor.

**Verification.** The cell below recomputes the boxed answer of Problem L3.7 and asserts agreement.

In [38]:
# First and second fundamental forms of a Monge patch, against finite differences.
f = lambda u, v: 0.4 * u**2 - 0.7 * u * v + 1.1 * np.sin(v)
fx = lambda u, v: 0.8 * u - 0.7 * v
fy = lambda u, v: -0.7 * u + 1.1 * np.cos(v)
fxx = lambda u, v: 0.8 + 0 * u
fxy = lambda u, v: -0.7 + 0 * u
fyy = lambda u, v: -1.1 * np.sin(v)

u0, v0 = 0.6, -0.9
gradf = np.array([fx(u0, v0), fy(u0, v0)])
Hf = np.array([[fxx(u0, v0), fxy(u0, v0)], [fxy(u0, v0), fyy(u0, v0)]])

r_u = np.array([1.0, 0.0, fx(u0, v0)])
r_v = np.array([0.0, 1.0, fy(u0, v0)])
I_direct = np.array([[r_u @ r_u, r_u @ r_v], [r_u @ r_v, r_v @ r_v]])
I_formula = np.eye(2) + np.outer(gradf, gradf)
report("L3.7  I vs I_2 + grad f grad f^T", I_direct, I_formula, tol=1e-13)

nvec = np.cross(r_u, r_v); nvec /= np.linalg.norm(nvec)
r_uu, r_uv, r_vv = np.array([0, 0, fxx(u0, v0)]), np.array([0, 0, fxy(u0, v0)]), np.array([0, 0, fyy(u0, v0)])
II_direct = np.array([[r_uu @ nvec, r_uv @ nvec], [r_uv @ nvec, r_vv @ nvec]])
II_formula = Hf / np.sqrt(1 + gradf @ gradf)
report("L3.7  II vs H_f / sqrt(1 + ||grad f||^2)", II_direct, II_formula, tol=1e-13)
print("I  =\n", I_direct, "\nII =\n", II_direct)

# Consequence: K = det(II)/det(I) = det(H)/(1 + ||grad f||^2)^2.
K_shape = np.linalg.det(II_direct) / np.linalg.det(I_direct)
K_monge = np.linalg.det(Hf) / (1 + gradf @ gradf) ** 2
report("L3.7  det(II)/det(I) vs det(H)/(1+||grad f||^2)^2", K_shape, K_monge, tol=1e-13)
print(f"\nGaussian curvature at ({u0}, {v0}) = {K_shape:+.6f}")

L3.7  I vs I_2 + grad f grad f^T                     residual = 0.000e+00
L3.7  II vs H_f / sqrt(1 + ||grad f||^2)             residual = 0.000e+00
I  =
 [[2.2321 0.2928]
 [0.2928 1.0696]] 
II =
 [[ 0.5273 -0.4614]
 [-0.4614  0.568 ]]
L3.7  det(II)/det(I) vs det(H)/(1+||grad f||^2)^2    residual = 1.388e-17

Gaussian curvature at (0.6, -0.9) = +0.037625


---

### Problem L3.8 — Rigorous $\varepsilon$-$\delta$ Proof of Schwarz's Theorem
**Source:** Spivak, *Calculus* (Ch. 5) & Kaczor & Nowak, *Problems in Mathematical Analysis*.  
**Problem Statement:**  
Let $f: \mathbb{R}^2 \to \mathbb{R}$ have continuous second partial derivatives $f_{xy}$ and $f_{yx}$ at $(x_0, y_0)$. Prove rigorously using the Mean Value Theorem that for any $\varepsilon \gt 0$, there exists $\delta \gt 0$ such that for all $\lvert h \rvert, \lvert k \rvert \lt \delta$:

$$
\left\vert \frac{f(x_0+h, y_0+k) - f(x_0+h, y_0) - f(x_0, y_0+k) + f(x_0, y_0)}{h k} - f_{xy}(x_0, y_0) \right\vert \lt \varepsilon
$$

#### First-Principles Intuition
We construct a double difference quotient over a rectangle of size $h \times k$. Applying the single-variable MVT twice reduces the double difference to $f_{xy}(\xi, \eta)$, which converges to $f_{xy}(x_0, y_0)$ by continuity.

#### Step-by-Step Solution
1. **Define Double Difference Function:**

$$
   \Delta(h, k) = f(x_0+h, y_0+k) - f(x_0+h, y_0) - f(x_0, y_0+k) + f(x_0, y_0)
$$

2. **First MVT Application:**
   Set $\phi(x) = f(x, y_0+k) - f(x, y_0)$. Then $\Delta(h, k) = \phi(x_0+h) - \phi(x_0)$.  
   By MVT, there exists $x_1$ between $x_0$ and $x_0+h$ such that:

$$
   \Delta(h, k) = h \phi'(x_1) = h \left[ f_x(x_1, y_0+k) - f_x(x_1, y_0) \right]
$$

3. **Second MVT Application:**
   Set $\psi(y) = f_x(x_1, y)$. The term in brackets is $\psi(y_0+k) - \psi(y_0)$.  
   By MVT on $\psi(y)$, there exists $y_1$ between $y_0$ and $y_0+k$ such that:

$$
   f_x(x_1, y_0+k) - f_x(x_1, y_0) = k \psi'(y_1) = k f_{yx}(x_1, y_1)
$$

   Thus:

$$
   \frac{\Delta(h, k)}{h k} = f_{yx}(x_1, y_1)
$$

4. **Apply $\varepsilon$-$\delta$ Continuity:**
   Since $f_{yx}$ is continuous at $(x_0, y_0)$, for any $\varepsilon \gt 0$, there exists $\delta \gt 0$ such that for all $(x, y)$ satisfying $\lVert (x,y) - (x_0,y_0)\rVert  \lt \delta$:

$$
   \lvert f_{yx}(x, y) - f_{yx}(x_0, y_0) \rvert \lt \varepsilon
$$

   Since $\lvert x_1 - x_0 \rvert \lt \lvert h \rvert$ and $\lvert y_1 - y_0 \rvert \lt \lvert k \rvert$, taking $\lvert h \rvert, \lvert k \rvert \lt \delta' := \delta/\sqrt{2}$ gives $\lVert (x_1,y_1) - (x_0,y_0) \rVert \lt \delta$, hence

$$
   \left\vert \frac{\Delta(h, k)}{h k} - f_{yx}(x_0, y_0) \right\vert = \lvert f_{yx}(x_1, y_1) - f_{yx}(x_0, y_0) \rvert \lt \varepsilon .
$$

5. **The Symmetric Grouping — this is what makes the argument non-circular:**
   Step 4 alone gives one limit; it does **not** by itself say anything about $f_{xy}$, and asserting $f_{xy} = f_{yx}$ at this point would assume the conclusion. Repeat steps 2–4 with the roles of the variables exchanged. The **same** quantity $\Delta(h,k)$ regroups as

$$
   \Delta(h,k) = \tilde{\phi}(y_0 + k) - \tilde{\phi}(y_0), \qquad \tilde{\phi}(y) := f(x_0 + h, y) - f(x_0, y),
$$

   because both expressions expand to the same four values of $f$ at the corners of the rectangle. Two applications of the mean value theorem — first to $\tilde{\phi}$ in $y$, then to $y \mapsto f_y(\cdot, y_2)$ in $x$ — produce points $x_2$ between $x_0$ and $x_0 + h$ and $y_2$ between $y_0$ and $y_0 + k$ with

$$
   \frac{\Delta(h,k)}{hk} = f_{xy}(x_2, y_2).
$$

   Continuity of $f_{xy}$ at $(x_0, y_0)$ gives, for $\lvert h \rvert, \lvert k \rvert$ small enough,

$$
   \left\vert \frac{\Delta(h,k)}{hk} - f_{xy}(x_0, y_0) \right\vert \lt \varepsilon .
$$

6. **Conclude:**
   For every $\varepsilon \gt 0$ there are $h, k$ small enough that both displays hold simultaneously for the *same* number $\Delta(h,k)/(hk)$, so by the triangle inequality

$$
   \lvert f_{xy}(x_0, y_0) - f_{yx}(x_0, y_0) \rvert \lt 2\varepsilon .
$$

   The left-hand side is a fixed number independent of $\varepsilon$, so it must be $0$.

$$
\boxed{\lim_{(h,k)\to(0,0)} \frac{\Delta(h,k)}{hk} = f_{yx}(x_0, y_0) = f_{xy}(x_0, y_0)}
$$

#### Key Insight / Takeaway
One grouping of the double difference gives $f_{yx}$ at an interior point and the other gives $f_{xy}$; equality follows because both describe the *same* number, and continuity is exactly what transfers that shared value to the point $(x_0, y_0)$ itself.

**Verification.** The cell below recomputes the boxed answer of Problem L3.8 and asserts agreement.

In [39]:
# The double difference quotient equals BOTH mixed partials at interior points, so its
# single limit forces them equal. Checked on a C^2 function, and broken on the classic
# non-C^2 one to show the hypothesis is doing the work.
x, y = sp.symbols("x y", real=True)
g = sp.exp(x * y) + sp.sin(x + 2 * y) + x**3 * y - y**2 * x
g_n = sp.lambdify((x, y), g, "numpy")
fxy_n = sp.lambdify((x, y), sp.diff(g, x, y), "numpy")
fyx_n = sp.lambdify((x, y), sp.diff(g, y, x), "numpy")
x0, y0 = 0.3, -0.7
print("symbolic gap d2g/dxdy - d2g/dydx =", sp.simplify(sp.diff(g, x, y) - sp.diff(g, y, x)))
assert sp.simplify(sp.diff(g, x, y) - sp.diff(g, y, x)) == 0

print("\n   h=k        Delta/(hk)          f_xy(x0,y0)        error")
for s in (1e-2, 1e-3, 1e-4):
    D = (g_n(x0 + s, y0 + s) - g_n(x0 + s, y0) - g_n(x0, y0 + s) + g_n(x0, y0)) / s**2
    print(f"   {s:.0e}   {D:.12f}   {fxy_n(x0, y0):.12f}   {abs(D - fxy_n(x0, y0)):.3e}")
    err = abs(D - fxy_n(x0, y0))
assert err < 1e-3
report("L3.8  f_xy vs f_yx on a C^2 function", fxy_n(x0, y0), fyx_n(x0, y0), tol=1e-12)

# Drop continuity of the mixed partials and the two orders separate.
def fx_bad(a, b):
    r2 = a * a + b * b
    return -b if r2 == 0 else b * (a**4 + 4 * a**2 * b**2 - b**4) / r2**2
def fy_bad(a, b):
    r2 = a * a + b * b
    return a if r2 == 0 else a * (a**4 - 4 * a**2 * b**2 - b**4) / r2**2
e = 1e-5
f_xy_00 = (fx_bad(0.0, e) - fx_bad(0.0, -e)) / (2 * e)
f_yx_00 = (fy_bad(e, 0.0) - fy_bad(-e, 0.0)) / (2 * e)
print(f"\nnon-C^2 example: f_xy(0,0) = {f_xy_00:+.6f}, f_yx(0,0) = {f_yx_00:+.6f}, gap = {abs(f_xy_00 - f_yx_00):.6f}")
assert abs(abs(f_xy_00 - f_yx_00) - 2.0) < 1e-6

symbolic gap d2g/dxdy - d2g/dydx = 0

   h=k        Delta/(hk)          f_xy(x0,y0)        error
   1e-02   4.075175739925   4.092776274439   1.760e-02
   1e-03   4.091024383701   4.092776274439   1.752e-03
   1e-04   4.092601169070   4.092776274439   1.751e-04
L3.8  f_xy vs f_yx on a C^2 function                 residual = 0.000e+00

non-C^2 example: f_xy(0,0) = -1.000000, f_yx(0,0) = +1.000000, gap = 2.000000


---

### Problem L3.9 — Rank-Deficiency of Deep Neural Network Gauss-Newton Hessian
**Source:** Sagun et al. (2017, *Empirical Analysis of the Hessian of Over-parametrized Neural Networks*).  
**Problem Statement:**  
Let $f(x; W) \in \mathbb{R}^C$ be a neural network with $p$ parameters ($p \gg C$) trained with MSE loss $\mathcal{L}(W) = \frac{1}{2} \lVert f(x; W) - y\rVert^2$ on a single data point. Prove that the Gauss-Newton Hessian $H_{GN} = J^T J$ has rank at most $C$, and deduce the dimension of zero-eigenvalue flat directions.

#### First-Principles Intuition
The Gauss-Newton matrix is constructed as $J^T J$, where $J \in \mathbb{R}^{C \times p}$ is the network output Jacobian. By linear algebra rank inequalities, $\operatorname{rank}(J^T J) = \operatorname{rank}(J) \le \min(C, p) = C$.

#### Step-by-Step Solution
1. **Identify Matrix Dimensions:**
   The output dimension is $C$ (number of classes).  
   The parameter vector dimension is $p$ (number of weights), with $p \gg C$.  
   The Jacobian matrix of output logits with respect to parameters is $J \in \mathbb{R}^{C \times p}$.

2. **Compute Gauss-Newton Hessian:**

$$
   H_{GN} = J^T J \in \mathbb{R}^{p \times p}
$$

3. **Apply Rank Property:**
   For any matrix $A \in \mathbb{R}^{m \times n}$, $\operatorname{rank}(A^T A) = \operatorname{rank}(A)$.  
   Here, $J$ has $C$ rows and $p$ columns, so:

$$
   \operatorname{rank}(J) \le \min(C, p) = C
$$

   Therefore:

$$
   \operatorname{rank}(H_{GN}) = \operatorname{rank}(J^T J) = \operatorname{rank}(J) \le C
$$

4. **Compute Nullspace Dimension:**
   By the Rank-Nullity Theorem for $H_{GN} \in \mathbb{R}^{p \times p}$:

$$
   \operatorname{dim}(\operatorname{Null}(H_{GN})) = p - \operatorname{rank}(H_{GN}) \ge p - C
$$

$$
\boxed{\operatorname{rank}(H_{GN}) \le C \implies \text{at least } p - C \text{ zero-eigenvalue flat directions in parameter space}}
$$

#### Key Insight / Takeaway
Over-parameterized neural networks ($p \gg C$) possess vast continuous manifolds of zero-curvature flat directions ($p - C$ zero eigenvalues), forming flat minima.

**Verification.** The cell below recomputes the boxed answer of Problem L3.9 and asserts agreement.

In [40]:
# Gauss-Newton Hessian J^T J: rank at most C, so at least p - C flat directions.
C, p = 4, 50
J = rng.normal(size=(C, p))
H_gn = J.T @ J
r = np.linalg.matrix_rank(H_gn)
print(f"C = {C}, p = {p}:  rank(J) = {np.linalg.matrix_rank(J)}, rank(J^T J) = {r}")
assert r == np.linalg.matrix_rank(J) <= C
lam = np.linalg.eigvalsh(H_gn)
print(f"eigenvalues: {C} nonzero, the rest at machine noise")
print("  largest  :", lam[-C:])
print(f"  |largest of the remaining {p - C}| = {np.abs(lam[:p - C]).max():.3e}")
assert np.abs(lam[:p - C]).max() < 1e-8 * lam[-1]
print(f"\nnullspace dimension = {p - r} >= p - C = {p - C}")
assert p - r >= p - C

# Any direction in the nullspace leaves the Gauss-Newton model value unchanged.
ns = np.linalg.svd(J)[2][C:]                       # rows spanning null(J)
v = ns[0]
report("L3.9  J v for v in null(J)", J @ v, np.zeros(C), tol=1e-12)
report("L3.9  v^T H_GN v", float(v @ H_gn @ v), 0.0, tol=1e-12)

C = 4, p = 50:  rank(J) = 4, rank(J^T J) = 4
eigenvalues: 4 nonzero, the rest at machine noise
  largest  : [38.3992 39.993  55.6159 82.7857]
  |largest of the remaining 46| = 2.342e-14

nullspace dimension = 46 >= p - C = 46
L3.9  J v for v in null(J)                           residual = 6.731e-16
L3.9  v^T H_GN v                                     residual = 2.479e-16


---

### Problem L3.10 — Hessian Determinant Inequality for Log-Concave Density
**Source:** Polya & Szego, *Problems and Theorems in Analysis II* / Cover & Thomas.  
**Problem Statement:**  
A probability density $p(x) \gt 0$ on $\mathbb{R}^n$ is called **log-concave** if $\phi(x) = -\ln p(x)$ is convex. Show that the Hessian of $-\ln p(x)$ satisfies $\nabla^2 (-\ln p(x)) \succeq 0$, and prove the inequality:

$$
p(x) \nabla^2 p(x) \preceq \nabla p(x) \nabla p(x)^T
$$

#### First-Principles Intuition
Log-concavity ensures that probability density contours behave like multivariate Gaussians, where second derivatives of $-\ln p(x)$ yield positive semi-definite information matrices.

#### Step-by-Step Solution
Let $\phi(x) = -\ln p(x) \implies p(x) = e^{-\phi(x)}$.

1. **Compute Gradient $\nabla \phi(x)$:**

$$
   \nabla \phi(x) = -\frac{\nabla p(x)}{p(x)} \implies \nabla p(x) = -p(x) \nabla \phi(x)
$$

2. **Compute Hessian Matrix $\nabla^2 \phi(x)$:**
   Differentiate $\nabla \phi(x) = -p(x)^{-1} \nabla p(x)$ with respect to $x^T$:

$$
   \nabla^2 \phi(x) = -\nabla \left( \frac{\nabla p(x)}{p(x)} \right) = -\frac{\nabla^2 p(x)}{p(x)} + \frac{\nabla p(x) \nabla p(x)^T}{p(x)^2}
$$

3. **Apply Convexity Condition ($\nabla^2 \phi(x) \succeq 0$):**
   Since $p(x)$ is log-concave, $\phi(x) = -\ln p(x)$ is convex, so $\nabla^2 \phi(x) \succeq 0$:

$$
   -\frac{\nabla^2 p(x)}{p(x)} + \frac{\nabla p(x) \nabla p(x)^T}{p(x)^2} \succeq 0
$$

4. **Rearrange Matrix Inequality:**
   Multiply the inequality by $p(x)^2 \gt 0$:

$$
   -p(x) \nabla^2 p(x) + \nabla p(x) \nabla p(x)^T \succeq 0
$$

   Rearranging terms yields:

$$
   p(x) \nabla^2 p(x) \preceq \nabla p(x) \nabla p(x)^T
$$

$$
\boxed{\nabla^2 (-\ln p(x)) \succeq 0 \iff p(x) \nabla^2 p(x) \preceq \nabla p(x) \nabla p(x)^T}
$$

#### Key Insight / Takeaway
Log-concavity bounds the density Hessian $p(x) \nabla^2 p(x)$ by the outer product of its gradient $\nabla p \nabla p^T$, ensuring unimodal probability distributions.

**Verification.** The cell below recomputes the boxed answer of Problem L3.10 and asserts agreement.

In [41]:
# Log-concavity: Hessian of -ln p is PSD exactly when p grad^2 p <= grad p grad p^T.
x, y = sp.symbols("x y", real=True)
mu = sp.Matrix([0.3, -0.5])
Sig = sp.Matrix([[1.4, 0.4], [0.4, 0.9]])
v = sp.Matrix([x, y]) - mu
p_sym = sp.exp(-sp.Rational(1, 2) * (v.T * Sig.inv() * v)[0, 0]) / (2 * sp.pi * sp.sqrt(Sig.det()))
phi = sp.simplify(-sp.log(p_sym))
H_phi = sp.simplify(sp.hessian(phi, (x, y)))
print("Hessian of -ln p for a Gaussian =", H_phi.tolist())
assert sp.simplify(H_phi - Sig.inv()) == sp.zeros(2, 2)

# The rearranged inequality, numerically: grad p grad p^T - p grad^2 p = p^2 * grad^2 phi >= 0.
p_n = sp.lambdify((x, y), p_sym, "numpy")
gp = sp.lambdify((x, y), sp.Matrix([sp.diff(p_sym, x), sp.diff(p_sym, y)]), "numpy")
Hp = sp.lambdify((x, y), sp.hessian(p_sym, (x, y)), "numpy")
Sig_inv = np.array(Sig.inv(), dtype=float)
for a, b in [(0.0, 0.0), (1.2, -0.4), (-2.0, 1.5)]:
    g = np.array(gp(a, b), dtype=float).ravel()
    M = np.outer(g, g) - p_n(a, b) * np.array(Hp(a, b), dtype=float)
    lam = np.linalg.eigvalsh(M)
    print(f"  at ({a:+.1f}, {b:+.1f}): eigenvalues of grad p grad p^T - p grad^2 p = {lam}")
    assert lam.min() > -1e-14
    report("L3.10  M vs p^2 * Sigma^{-1}", M, p_n(a, b) ** 2 * Sig_inv, tol=1e-12)

# A density that is not log-concave violates it: p proportional to exp(-x^4 + 3x^2) in 1-D.
xs = sp.Symbol("xs", real=True)
q = sp.exp(-xs**4 + 3 * xs**2)
phi_q = sp.simplify(-sp.log(q))
d2 = sp.lambdify(xs, sp.diff(phi_q, xs, 2), "numpy")
print(f"\nnon-log-concave example: (-ln q)''(0) = {d2(0.0):+.4f} < 0, so the inequality fails there")
assert d2(0.0) < 0

Hessian of -ln p for a Gaussian = [[0.818181818181818, -0.363636363636364], [-0.363636363636364, 1.27272727272727]]
  at (+0.0, +0.0): eigenvalues of grad p grad p^T - p grad^2 p = [0.0086 0.0206]
L3.10  M vs p^2 * Sigma^{-1}                         residual = 1.475e-17
  at (+1.2, -0.4): eigenvalues of grad p grad p^T - p grad^2 p = [0.0077 0.0184]
L3.10  M vs p^2 * Sigma^{-1}                         residual = 1.301e-17
  at (-2.0, +1.5): eigenvalues of grad p grad p^T - p grad^2 p = [0. 0.]
L3.10  M vs p^2 * Sigma^{-1}                         residual = 3.044e-22

non-log-concave example: (-ln q)''(0) = -6.0000 < 0, so the inequality fails there


---